In [2]:
"""
repair_bengaluru_aq.py
======================
Physical-consistency and instrument-fault repair pass for the Bengaluru
CPCB combined dataset.

INPUT   Bengaluru_AirQuality_combined.csv   (read-only; never modified)
OUTPUT  Bengaluru_AirQuality_repaired.csv   (new file)
        repair_audit_log.csv                (one row per changed/flagged cell)

DESIGN PRINCIPLE
----------------
Two mechanisms, deliberately kept separate:

  REPAIR  - short, isolated, physically impossible values are re-nulled and
            re-imputed with the same transparent interpolate -> ffill/bfill
            logic used in the original preprocessing notebook.

  FLAG    - long instrument-fault episodes are NOT imputed. Filling weeks or
            months of a series from its own neighbours manufactures signal.
            These keep their original value and receive a quality flag so they
            can be masked during model fitting and evaluation.

Every original value is preserved in a `<pollutant>_raw` column, so nothing
is destroyed and every decision is reversible.

Author: prepared for the AirSense-IQ Bengaluru project.
"""

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Configuration - every threshold is here, and every one is a judgement call
# that should be reported in the methodology section.
# ----------------------------------------------------------------------------
IN_CSV = "../data/processed/Bengaluru_AirQuality_combined.csv"
OUT_CSV = "../data/processed/Bengaluru_AirQuality_repaired.csv"
AUDIT_CSV = "repair_audit_log.csv"

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# PM2.5/PM10 ratio bands. PM10 includes PM2.5 by definition, so ratio > 1 is
# impossible. But near 1.0 it is within the combined uncertainty of two
# independent instruments, so those are flagged rather than repaired.
PM_RATIO_IMPOSSIBLE = 1.50   # above this: PM2.5 is the faulty channel
PM_RATIO_AMBIGUOUS = 1.15    # between: cannot attribute, null both
                             # below: measurement noise, flag only

FLATLINE_MIN = 5             # identical consecutive values = stuck sensor
FLATLINE_REPAIRABLE_MAX = 14 # longer runs are flagged, not imputed

EPISODE_MIN_DAYS = 5         # sustained cross-station divergence
EPISODE_Z_HIGH = 3.0         # relative to the station's OWN typical city ratio
EPISODE_Z_LOW = 1 / 3.0
SPIKE_Z = 5.0                # isolated single/short-run cross-station spike

# Minimum credible city-wide level. Below this the cross-station control is
# itself untrustworthy, so divergence tests are suppressed.
CITY_FLOOR = {"PM2.5": 2.0, "PM10": 5.0, "NO2": 2.0,
              "SO2": 1.0, "CO": 0.10, "Ozone": 2.0}

REGIME_MIN_MONTHS = 2        # instrument regime shifts live at monthly scale
REGIME_Z_HIGH = 2.0
REGIME_Z_LOW = 0.5

# Network-wide CO failure window (see data-quality report for the evidence).
CO_FAILURE_FROM = pd.Timestamp("2026-01-01")


# ----------------------------------------------------------------------------
def load(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    return df.sort_values(["Station", "Date"]).reset_index(drop=True)


def relative_to_city(df, pollutant):
    """Each station's value divided by the median of the OTHER five stations
    that day, then divided by that station's own long-run median of the same
    quantity. Result ~1 means 'behaving as this station normally behaves
    relative to the city'; the normalisation is what stops a permanently
    dirtier station (Silk Board PM10) being flagged as faulty."""
    wide = df.pivot(index="Date", columns="Station", values=pollutant)
    loo = pd.DataFrame(
        {s: wide.drop(columns=s).median(axis=1) for s in wide.columns}
    )
    floor = CITY_FLOOR[pollutant]
    rel = wide / loo.where(loo > floor)
    return rel / rel.median(), wide, loo


def runs_of(mask_series):
    """Yield (start_idx, end_idx, length) for each run of True."""
    v = np.asarray(mask_series, dtype=bool)
    i = 0
    while i < len(v):
        if v[i]:
            j = i
            while j + 1 < len(v) and v[j + 1]:
                j += 1
            yield i, j, j - i + 1
            i = j + 1
        else:
            i += 1


def flatline_runs(values, already_imputed):
    """Runs of >= FLATLINE_MIN identical consecutive values that the original
    pipeline did not already mark as imputed."""
    v = np.asarray(values, dtype=float)
    imp = np.asarray(already_imputed, dtype=bool)
    out, i = [], 0
    while i < len(v):
        j = i
        while j + 1 < len(v) and v[j + 1] == v[i]:
            j += 1
        length = j - i + 1
        if length >= FLATLINE_MIN and not imp[i:j + 1].any():
            out.append((i, j, length))
        i = j + 1
    return out


# ----------------------------------------------------------------------------
def build_quality_masks(df):
    """Return `quality`: a DataFrame of strings, one column per pollutant,
    with '' meaning no issue detected."""
    q = pd.DataFrame("", index=df.index, columns=POLLUTANTS)

    # --- Rule A: PM2.5 / PM10 physical inconsistency -------------------------
    ratio = df["PM2.5"] / df["PM10"].replace(0, np.nan)
    q.loc[ratio > PM_RATIO_IMPOSSIBLE, "PM2.5"] = "repair:pm_ratio_impossible"
    amb = (ratio > PM_RATIO_AMBIGUOUS) & (ratio <= PM_RATIO_IMPOSSIBLE)
    q.loc[amb, "PM2.5"] = "repair:pm_ratio_ambiguous"
    q.loc[amb, "PM10"] = "repair:pm_ratio_ambiguous"
    marg = (ratio > 1.0) & (ratio <= PM_RATIO_AMBIGUOUS)
    q.loc[marg, "PM2.5"] = "flag:pm_ratio_marginal"

    # --- Rule B: CO exact-zero sentinel --------------------------------------
    # Ambient CO is never exactly 0.00; this is a missing-data placeholder.
    zero_co = df["CO"] == 0.0
    q.loc[zero_co, "CO"] = "repair:co_zero_sentinel"

    # --- Rule C: network-wide CO failure from 2026 ---------------------------
    # Overrides the repair above: in this window the neighbouring stations are
    # themselves degraded, so there is nothing trustworthy to impute from.
    co_fail = df["Date"] >= CO_FAILURE_FROM
    q.loc[co_fail, "CO"] = "unusable:co_network_failure_2026"

    # --- Rule D: stuck-sensor flatlines --------------------------------------
    for p in POLLUTANTS:
        for st, g in df.groupby("Station"):
            for i, j, length in flatline_runs(g[p].values,
                                              g[f"{p}_is_imputed"].values):
                idx = g.index[i:j + 1]
                tag = ("repair:flatline" if length <= FLATLINE_REPAIRABLE_MAX
                       else "unusable:flatline_long")
                blank = q.loc[idx, p] == ""
                q.loc[idx[blank], p] = tag

    # --- Rule E: cross-station divergence ------------------------------------
    for p in POLLUTANTS:
        z, wide, _ = relative_to_city(df, p)
        key = df.set_index(["Date", "Station"]).index
        for st in z.columns:
            s = z[st]
            sustained = ((s > EPISODE_Z_HIGH) | (s < EPISODE_Z_LOW)).fillna(False)
            spike = ((s > SPIKE_Z) | (s < 1 / SPIKE_Z)).fillna(False)
            for i, j, length in runs_of(sustained):
                dates = s.index[i:j + 1]
                if length >= EPISODE_MIN_DAYS:
                    tag = "unusable:station_divergence"
                elif spike.iloc[i:j + 1].any():
                    tag = "repair:cross_station_spike"
                else:
                    continue
                pos = key.get_indexer(pd.MultiIndex.from_product([dates, [st]]))
                pos = pos[pos >= 0]
                blank = q.iloc[pos][p] == ""
                q.iloc[pos[blank.values], q.columns.get_loc(p)] = tag

    # --- Rule F: monthly regime shifts ---------------------------------------
    # A daily detector under-catches an analyser that sits in a wrong state for
    # months, because its own baseline gets contaminated by the fault. The same
    # normalised comparison at monthly resolution finds these. This is what
    # identifies the Hebbal NO2 analyser switching between a stuck-low (~4) and
    # a stuck-high (~50) state across 2023-2025.
    month = df["Date"].dt.to_period("M")
    for p in POLLUTANTS:
        wide = df.pivot(index="Date", columns="Station", values=p) \
                 .resample("MS").median()
        loo = pd.DataFrame(
            {s: wide.drop(columns=s).median(axis=1) for s in wide.columns}
        )
        rel = wide / loo.where(loo > CITY_FLOOR[p])
        z = rel / rel.median()
        for st in wide.columns:
            bad = ((z[st] > REGIME_Z_HIGH) | (z[st] < REGIME_Z_LOW)).fillna(False)
            for i, j, length in runs_of(bad.values):
                if length < REGIME_MIN_MONTHS:
                    continue
                periods = wide.index[i:j + 1].to_period("M")
                sel = (df["Station"] == st) & month.isin(periods)
                blank = sel & (q[p] == "")
                q.loc[blank, p] = "unusable:regime_shift"

    return q


# ----------------------------------------------------------------------------
def reimpute(df, quality):
    """Null every cell tagged `repair:*`, then re-apply the original
    interpolate(time) -> ffill -> bfill logic per station. `unusable:*` and
    `flag:*` cells keep their value untouched."""
    out = df.copy()
    for p in POLLUTANTS:
        out[f"{p}_raw"] = df[p]

    to_null = quality.apply(lambda c: c.str.startswith("repair:"))
    for p in POLLUTANTS:
        out.loc[to_null[p], p] = np.nan

    frames = []
    for st, g in out.groupby("Station", sort=False):
        g = g.sort_values("Date").set_index("Date")
        g[POLLUTANTS] = g[POLLUTANTS].interpolate(method="time",
                                                  limit_direction="both")
        g[POLLUTANTS] = g[POLLUTANTS].ffill().bfill()
        frames.append(g.reset_index())
    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["Station", "Date"]).reset_index(drop=True)
    return out, to_null


def rebuild_flags(out, df, quality, to_null):
    """`_is_imputed` becomes the union of original source gaps and newly
    repaired cells. `_impute_run_len` is recomputed on that union so a repaired
    cell adjacent to a source gap reports the true length of the estimated
    stretch. `_quality` records why."""
    for p in POLLUTANTS:
        est = df[f"{p}_is_imputed"].astype(bool) | to_null[p].values
        out[f"{p}_is_imputed"] = est

        runlen = pd.Series(0, index=out.index)
        for st, g in out.groupby("Station", sort=False):
            m = est.loc[g.index]
            for i, j, length in runs_of(m.values):
                runlen.loc[g.index[i:j + 1]] = length
        out[f"{p}_impute_run_len"] = runlen

        qual = quality[p].copy()
        qual[(qual == "") & df[f"{p}_is_imputed"].astype(bool)] = "imputed:source_gap"
        qual[qual == ""] = "measured"
        out[f"{p}_quality"] = qual.values
    return out


# ----------------------------------------------------------------------------
def validate(out, df):
    problems = []
    if out.shape[0] != df.shape[0]:
        problems.append(f"row count changed: {df.shape[0]} -> {out.shape[0]}")
    if out[POLLUTANTS].isna().any().any():
        problems.append("nulls remain in pollutant columns")
    for p in POLLUTANTS:
        if not np.allclose(out[f"{p}_raw"], df[p], equal_nan=True):
            problems.append(f"{p}_raw does not match the original input")
        untouched = out[f"{p}_quality"].isin(["measured", "flag:pm_ratio_marginal"])
        if not np.allclose(out.loc[untouched, p], out.loc[untouched, f"{p}_raw"]):
            problems.append(f"{p}: a 'measured' cell was modified")
    if out.duplicated(["Station", "Date"]).any():
        problems.append("duplicate (Station, Date) introduced")
    if (out[POLLUTANTS] < 0).any().any():
        problems.append("negative concentration produced")
    return problems


def main():
    df = load(IN_CSV)
    quality = build_quality_masks(df)
    out, to_null = reimpute(df, quality)
    out = rebuild_flags(out, df, quality, to_null)

    cols = ["Date", "Station"]
    for p in POLLUTANTS:
        cols += [p, f"{p}_raw", f"{p}_quality",
                 f"{p}_is_imputed", f"{p}_impute_run_len"]
    out = out[cols]

    problems = validate(out, df)
    print("VALIDATION:", "all checks passed" if not problems else problems)

    print("\nCells by quality class:")
    summary = pd.DataFrame({p: out[f"{p}_quality"].value_counts()
                            for p in POLLUTANTS}).fillna(0).astype(int)
    print(summary.to_string())

    audit = []
    for p in POLLUTANTS:
        m = ~out[f"{p}_quality"].isin(["measured"])
        for _, r in out.loc[m, ["Date", "Station", p, f"{p}_raw",
                                f"{p}_quality"]].iterrows():
            audit.append(dict(Date=r["Date"].date(), Station=r["Station"],
                              Pollutant=p, raw_value=r[f"{p}_raw"],
                              new_value=r[p], quality=r[f"{p}_quality"]))
    audit = pd.DataFrame(audit).sort_values(["Pollutant", "Station", "Date"])

    out.to_csv(OUT_CSV, index=False)
    audit.to_csv(AUDIT_CSV, index=False)
    print(f"\nWrote {OUT_CSV}  {out.shape}")
    print(f"Wrote {AUDIT_CSV}  {audit.shape[0]} audited cells")
    return out, audit, summary


if __name__ == "__main__":
    main()

VALIDATION: all checks passed

Cells by quality class:
                                  PM2.5  PM10   NO2   SO2    CO  Ozone
flag:pm_ratio_marginal                6     0     0     0     0      0
imputed:source_gap                  359   371   430   369   423    474
measured                           5896  5980  4925  6020  4584   5634
repair:cross_station_spike           33    11    40    30    11     38
repair:flatline                       9    16     0    48   100     15
repair:pm_ratio_ambiguous             6     6     0     0     0      0
repair:pm_ratio_impossible           13     0     0     0     0      0
unusable:co_network_failure_2026      0     0     0     0  1086      0
unusable:flatline_long                0     0     0     0     0     45
unusable:regime_shift               208   177   723    44   328    303
unusable:station_divergence          46    15   458    65    44     67

Wrote Bengaluru_AirQuality_repaired.csv  (6576, 32)
Wrote repair_audit_log.csv  6417 audited

In [4]:
"""
fetch_weather_bengaluru.py
==========================
Stage 1 of 2. Downloads hourly historical weather for the six Bengaluru CAAQMS
station coordinates from the Open-Meteo Historical Weather API and writes ONE
raw hourly CSV. No aggregation, no merging, no cleaning happens here — that is
stage 2 (aggregate_and_merge_weather.py), so the slow network step is run once
and the fast analysis step can be re-run freely.

RUN THIS FIRST.  Requires: pip install requests pandas

    python fetch_weather_bengaluru.py

OUTPUT
    Bengaluru_weather_hourly_raw.csv    ~157,824 rows (6 stations x 26,304 h)
    weather_source_provenance.csv       what the API actually returned per station

SOURCE
    Open-Meteo Historical Weather API, https://archive-api.open-meteo.com/v1/archive
    Reanalysis: ECMWF IFS 9 km (2017-present) via models=best_match, which
    combines IFS HRES, ERA5 and ERA5-Land. The study window sits entirely after
    2017, so IFS covers it throughout with no mid-series model change.
    Licence CC BY 4.0. No API key required.

    Cite: Zippenfenig, P. (2023). Open-Meteo.com Weather API. Zenodo.
          https://doi.org/10.5281/ZENODO.7970649
          Hersbach, H. et al. (2023). ERA5 hourly data on single levels from
          1940 to present. ECMWF. https://doi.org/10.24381/cds.adbb2d47

STATION COORDINATES
    Report of the Joint Committee (MoEF&CC / CPCB / KSPCB), Annexure I to CPCB's
    reply in NGT OA No. 15 of 2025 (SZ), Table 1: Operational Status of CAAQMS
    in Bengaluru.
"""

import sys
import time

import pandas as pd
import requests

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"

START_DATE = "2023-07-01"
END_DATE = "2026-06-30"
TIMEZONE = "Asia/Kolkata"        # returns local-time stamps starting 00:00 IST
MODEL = "best_match"             # set "era5" for the consistency cross-check

OUT_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw.csv"
OUT_PROVENANCE = "../data/processed/weather_source_provenance.csv"

# Station name -> (latitude, longitude, official CAAQMS name, agency)
STATIONS = {
    "BTM Layout": (
        12.9135218, 77.5950804,
        "BTM Layout, Bengaluru", "CPCB"),
    "Hebbal": (
        13.029152, 77.585901,
        "Veterinary College, Hebbal, Opp. CBI Office", "KSPCB"),
    "Jayanagar 5th Block": (
        12.920984, 77.584908,
        "Shalini Ground, 541, 10th Main Rd, 5th Block, Jayanagar", "KSPCB"),
    "Kasturi Nagar": (
        13.004533, 77.664561,
        "RTO Office, Indiranagara, Bengaluru", "KSPCB"),
    "Peenya": (
        13.0270199, 77.494094,
        "Peenya Industrial Area, Shivapura, Peenya", "CPCB"),
    "Silk Board": (
        12.917348, 77.622813,
        "Central Silk Board, Silkboard Junction, HSR Layout 5th Sector", "KSPCB"),
}

# Requested hourly fields. Order is irrelevant for the JSON API (each variable
# comes back keyed by name), but is kept meaningful for readability.
#
#   INSTANTANEOUS  valid at the stamped hour
#   ACCUMULATED    covers the hour PRECEDING the stamp -- this matters for the
#                  daily aggregation and is handled in stage 2, not here.
HOURLY_INSTANT = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "boundary_layer_height",
]
HOURLY_ACCUMULATED = [
    "precipitation",
    "shortwave_radiation",
]
HOURLY = HOURLY_INSTANT + HOURLY_ACCUMULATED

# m/s rather than the km/h default: ventilation index (PBL height x wind speed)
# then comes out in m^2/s, the form used in the air-quality literature.
UNITS = {"wind_speed_unit": "ms", "temperature_unit": "celsius",
         "precipitation_unit": "mm", "timeformat": "iso8601"}

REQUEST_TIMEOUT = 90
MAX_RETRIES = 4
PAUSE_BETWEEN_STATIONS = 1.5     # be polite to a free public API


# ----------------------------------------------------------------------------
def fetch_station(name, lat, lon):
    """One request per station. Returns (hourly DataFrame, provenance dict)."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(HOURLY),
        "timezone": TIMEZONE,
        "models": MODEL,
        **UNITS,
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(ENDPOINT, params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code == 400:
                # Open-Meteo returns a descriptive JSON error for bad parameters
                # (e.g. a misspelled variable). Retrying will not help.
                raise SystemExit(
                    f"\n[FATAL] API rejected the request for {name}:\n"
                    f"        {r.json().get('reason', r.text)}\n"
                    f"        Check the variable names in HOURLY against\n"
                    f"        https://open-meteo.com/en/docs/historical-weather-api"
                )
            r.raise_for_status()
            payload = r.json()
            break
        except requests.RequestException as exc:
            last_error = exc
            wait = 2 ** attempt
            print(f"  attempt {attempt}/{MAX_RETRIES} failed ({exc}); "
                  f"retrying in {wait}s")
            time.sleep(wait)
    else:
        raise SystemExit(f"\n[FATAL] {name}: all retries failed. Last error:\n"
                         f"        {last_error}")

    hourly = pd.DataFrame(payload["hourly"])
    hourly["time"] = pd.to_datetime(hourly["time"])   # naive local (IST) stamps
    hourly.insert(0, "Station", name)

    missing = [v for v in HOURLY if v not in hourly.columns]
    if missing:
        print(f"  [WARN] {name}: model returned no data for {missing}")

    provenance = {
        "Station": name,
        "requested_lat": lat,
        "requested_lon": lon,
        "grid_lat": payload.get("latitude"),
        "grid_lon": payload.get("longitude"),
        "grid_elevation_m": payload.get("elevation"),
        "timezone": payload.get("timezone"),
        "utc_offset_seconds": payload.get("utc_offset_seconds"),
        "model": MODEL,
        "n_hours": len(hourly),
    }
    for var in HOURLY:
        provenance[f"unit_{var}"] = payload.get("hourly_units", {}).get(var)
        provenance[f"nulls_{var}"] = (
            int(hourly[var].isna().sum()) if var in hourly.columns else None)

    return hourly, provenance


# ----------------------------------------------------------------------------
def validate(hourly, provenance):
    """Fail loudly rather than silently shipping a short or shifted series."""
    problems = []

    expected_hours = (pd.Timestamp(END_DATE) - pd.Timestamp(START_DATE)).days + 1
    expected_hours *= 24

    for _, p in provenance.iterrows():
        if p["n_hours"] != expected_hours:
            problems.append(
                f"{p['Station']}: got {p['n_hours']} hours, expected {expected_hours}")
        if p["utc_offset_seconds"] != 19800:      # IST = UTC+5:30
            problems.append(
                f"{p['Station']}: utc_offset is {p['utc_offset_seconds']}s, "
                f"expected 19800 (IST). Daily aggregation would be misaligned.")

    for station, g in hourly.groupby("Station"):
        t = g["time"].sort_values()
        if t.min() != pd.Timestamp(f"{START_DATE} 00:00"):
            problems.append(f"{station}: series starts at {t.min()}, not 00:00 IST")
        gaps = t.diff().dropna().value_counts()
        if list(gaps.index) != [pd.Timedelta(hours=1)]:
            problems.append(f"{station}: irregular hourly spacing {dict(gaps)}")

    if "boundary_layer_height" in hourly.columns:
        if hourly["boundary_layer_height"].isna().all():
            problems.append(
                "boundary_layer_height is entirely null. This model may not "
                "supply it — try models=era5, or drop PBL and the ventilation "
                "index from the variable set and document the change.")

    return problems


def main():
    print(f"Open-Meteo archive | model={MODEL} | {START_DATE} to {END_DATE} "
          f"| timezone={TIMEZONE}\n")

    frames, prov = [], []
    for name, (lat, lon, official, agency) in STATIONS.items():
        print(f"Fetching {name} ({lat}, {lon}) ...")
        hourly, p = fetch_station(name, lat, lon)
        p["official_caaqms_name"] = official
        p["monitoring_agency"] = agency
        frames.append(hourly)
        prov.append(p)
        offset_km = 111.0 * (
            (p["grid_lat"] - lat) ** 2 + (p["grid_lon"] - lon) ** 2) ** 0.5
        print(f"  {len(hourly):,} hours | grid cell "
              f"({p['grid_lat']:.4f}, {p['grid_lon']:.4f}) "
              f"~{offset_km:.1f} km from station | "
              f"elevation {p['grid_elevation_m']} m")
        time.sleep(PAUSE_BETWEEN_STATIONS)

    hourly = pd.concat(frames, ignore_index=True)
    provenance = pd.DataFrame(prov)

    problems = validate(hourly, provenance)
    print("\nVALIDATION:", "all checks passed" if not problems else "")
    for p in problems:
        print("  [FAIL]", p)
    if problems:
        print("\nNot writing output. Fix the above and re-run.")
        sys.exit(1)

    hourly.to_csv(OUT_HOURLY, index=False)
    provenance.to_csv(OUT_PROVENANCE, index=False)
    print(f"\nWrote {OUT_HOURLY}  {hourly.shape}")
    print(f"Wrote {OUT_PROVENANCE}  {provenance.shape}")

    print("\nHow many DISTINCT grid cells the six stations resolved to:")
    cells = provenance.groupby(["grid_lat", "grid_lon"])["Station"].apply(list)
    for (la, lo), members in cells.items():
        print(f"  ({la}, {lo}): {members}")
    if len(cells) < len(STATIONS):
        print("  ^ Stations sharing a cell will have IDENTICAL weather columns.\n"
              "    This is a spatial-resolution limit of the reanalysis, not a\n"
              "    bug. Record it as a limitation.")


if __name__ == "__main__":
    main()

Open-Meteo archive | model=best_match | 2023-07-01 to 2026-06-30 | timezone=Asia/Kolkata

Fetching BTM Layout (12.9135218, 77.5950804) ...
  26,304 hours | grid cell (12.8998, 77.5749) ~2.7 km from station | elevation 915.0 m
Fetching Hebbal (13.029152, 77.585901) ...
  26,304 hours | grid cell (13.0404, 77.5523) ~3.9 km from station | elevation 921.0 m
Fetching Jayanagar 5th Block (12.920984, 77.584908) ...
  26,304 hours | grid cell (12.8998, 77.5749) ~2.6 km from station | elevation 915.0 m
Fetching Kasturi Nagar (13.004533, 77.664561) ...
  26,304 hours | grid cell (12.9701, 77.6455) ~4.4 km from station | elevation 905.0 m
Fetching Peenya (13.0270199, 77.494094) ...
  26,304 hours | grid cell (13.0404, 77.4704) ~3.0 km from station | elevation 898.0 m
Fetching Silk Board (12.917348, 77.622813) ...
  26,304 hours | grid cell (12.8998, 77.6567) ~4.2 km from station | elevation 884.0 m

VALIDATION: all checks passed

Wrote Bengaluru_weather_hourly_raw.csv  (157824, 12)
Wrote weather_

In [5]:
"""
aggregate_and_merge_weather.py
==============================
Stage 2 of 2. Aggregates the raw hourly weather to daily IST calendar days and
merges it onto the repaired air-quality panel. Run after
fetch_weather_bengaluru.py.

    python aggregate_and_merge_weather.py

INPUTS
    Bengaluru_weather_hourly_raw.csv       from stage 1
    weather_source_provenance.csv          from stage 1
    Bengaluru_AirQuality_repaired.csv      from the repair pass

OUTPUT
    Bengaluru_weather_daily.csv            6,576 rows, weather only
    Bengaluru_AQ_Weather_daily.csv         6,576 rows, the merged panel

No feature engineering happens here. Lags, rolling windows, cyclical encodings
and interaction terms are the next step and are deliberately kept separate so
that leakage can be reasoned about against a clean contemporaneous baseline.

TWO THINGS THIS FILE GETS RIGHT THAT A NAIVE GROUPBY WOULD NOT
--------------------------------------------------------------
1. Accumulated variables are stamped at the END of the hour they describe.
   Open-Meteo returns precipitation as the "preceding hour sum" and shortwave
   radiation as the "preceding hour mean", so the value stamped 2024-03-05
   00:00 actually describes 23:00-00:00 on 2024-03-04. Assigning it to the 5th
   would leak an hour of the previous day into every daily total. These
   variables are therefore shifted back one hour before the date is taken.
   Instantaneous variables (temperature, wind, pressure, humidity, cloud, PBL)
   are valid AT the stamp and are dated as-is.

2. Wind direction is averaged as a VECTOR, not arithmetically. The arithmetic
   mean of 350 deg and 10 deg is 180 deg - the exact opposite of the true
   resultant, which is 0 deg. Directions are decomposed into speed-weighted u/v
   components, those are averaged, and the resultant direction is recovered.
"""

import numpy as np
import pandas as pd

IN_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw.csv"
IN_PROVENANCE = "../data/processed/weather_source_provenance.csv"
IN_AQ = "../data/processed/Bengaluru_AirQuality_repaired.csv"

OUT_WEATHER_DAILY = "../data/processed/Bengaluru_weather_daily.csv"
OUT_MERGED = "../data/processed/Bengaluru_AQ_Weather_daily.csv"

STUDY_START = pd.Timestamp("2023-07-01")
STUDY_END = pd.Timestamp("2026-06-30")

# Variables stamped at the end of the hour they describe (see docstring note 1)
ACCUMULATED = ["precipitation", "shortwave_radiation"]

RAIN_HOUR_THRESHOLD_MM = 0.1     # Open-Meteo stores precipitation to 0.1 mm


# ----------------------------------------------------------------------------
def add_derived_hourly(h):
    """Quantities that must be computed hour by hour, before any averaging."""
    theta = np.deg2rad(h["wind_direction_10m"])

    # Meteorological convention: direction is where the wind blows FROM, so the
    # motion vector carries a negative sign. Components are speed-weighted,
    # which is what makes the daily resultant a transport direction rather than
    # a mean of bearings.
    h["wind_u"] = -h["wind_speed_10m"] * np.sin(theta)
    h["wind_v"] = -h["wind_speed_10m"] * np.cos(theta)

    # Ventilation index = mixing depth x transport speed. Averaged hourly and
    # then daily, NOT computed from daily means, because the product of means
    # is not the mean of products and the PBL/wind diurnal cycles are strongly
    # in phase (both peak mid-afternoon).
    h["ventilation_index"] = h["boundary_layer_height"] * h["wind_speed_10m"]

    return h


def assign_dates(h):
    """Date column per timing convention. Instantaneous -> stamp date.
    Accumulated -> date of the hour the value actually covers."""
    h["date_instant"] = h["time"].dt.normalize()
    h["date_accum"] = (h["time"] - pd.Timedelta(hours=1)).dt.normalize()
    return h


def aggregate(h):
    inst = h.groupby(["Station", "date_instant"]).agg(
        temp_mean=("temperature_2m", "mean"),
        temp_min=("temperature_2m", "min"),
        temp_max=("temperature_2m", "max"),
        rh_mean=("relative_humidity_2m", "mean"),
        rh_min=("relative_humidity_2m", "min"),
        rh_max=("relative_humidity_2m", "max"),
        dewpoint_mean=("dew_point_2m", "mean"),
        pressure_mean=("surface_pressure", "mean"),
        pressure_min=("surface_pressure", "min"),
        pressure_max=("surface_pressure", "max"),
        cloud_cover_mean=("cloud_cover", "mean"),
        wind_speed_mean=("wind_speed_10m", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        wind_u_mean=("wind_u", "mean"),
        wind_v_mean=("wind_v", "mean"),
        pbl_mean=("boundary_layer_height", "mean"),
        pbl_min=("boundary_layer_height", "min"),
        pbl_max=("boundary_layer_height", "max"),
        ventilation_index=("ventilation_index", "mean"),
        n_hours_instant=("temperature_2m", "size"),
    ).reset_index().rename(columns={"date_instant": "Date"})

    acc = h.groupby(["Station", "date_accum"]).agg(
        rain_sum=("precipitation", "sum"),
        rain_max_hourly=("precipitation", "max"),
        rain_hours=("precipitation",
                    lambda s: int((s > RAIN_HOUR_THRESHOLD_MM).sum())),
        # Hourly shortwave is a mean irradiance in W/m^2 over the preceding
        # hour, so energy for that hour is value * 3600 J/m^2. Summing over the
        # day and dividing by 1e6 gives the daily total in MJ/m^2.
        solar_rad_sum=("shortwave_radiation",
                       lambda s: s.sum() * 3600.0 / 1e6),
        solar_rad_max=("shortwave_radiation", "max"),
        n_hours_accum=("precipitation", "size"),
    ).reset_index().rename(columns={"date_accum": "Date"})

    daily = inst.merge(acc, on=["Station", "Date"], how="outer")

    # Derived from the averaged vector components, not from the raw bearings.
    daily["wind_speed_resultant"] = np.hypot(daily["wind_u_mean"],
                                             daily["wind_v_mean"])
    daily["wind_dir_resultant"] = (
        np.degrees(np.arctan2(-daily["wind_u_mean"], -daily["wind_v_mean"]))
        % 360)

    # Ratio of resultant to scalar mean speed: 1.0 means the wind held a
    # steady direction all day, near 0 means it boxed the compass and the
    # daily direction carries little information. Useful as a confidence
    # weight on any wind-direction feature built later.
    daily["wind_steadiness"] = (
        daily["wind_speed_resultant"] / daily["wind_speed_mean"]).where(
            daily["wind_speed_mean"] > 0)

    daily["temp_range"] = daily["temp_max"] - daily["temp_min"]
    daily["pressure_range"] = daily["pressure_max"] - daily["pressure_min"]

    return daily


def trim_and_check(daily):
    """The one-hour shift means the first and last day are partial: the day
    before STUDY_START picks up a single accumulated hour, and STUDY_END loses
    its final hour. Drop the out-of-window day and flag the incomplete one."""
    daily = daily[(daily["Date"] >= STUDY_START) &
                  (daily["Date"] <= STUDY_END)].copy()

    daily["weather_hours_complete"] = (
        (daily["n_hours_instant"] == 24) & (daily["n_hours_accum"] == 24))

    incomplete = daily.loc[~daily["weather_hours_complete"],
                           ["Station", "Date", "n_hours_instant",
                            "n_hours_accum"]]
    if len(incomplete):
        print(f"\n[NOTE] {len(incomplete)} station-days built from fewer than "
              f"24 hours (expected: the final day loses its 24th accumulated "
              f"hour to the timing shift):")
        print(incomplete.to_string(index=False))

    return daily


def merge_with_aq(daily, provenance):
    aq = pd.read_csv(IN_AQ, parse_dates=["Date"])

    prov_cols = provenance[["Station", "grid_lat", "grid_lon",
                            "grid_elevation_m", "model"]].rename(columns={
                                "grid_lat": "weather_grid_lat",
                                "grid_lon": "weather_grid_lon",
                                "grid_elevation_m": "weather_grid_elevation_m",
                                "model": "weather_model"})
    daily = daily.merge(prov_cols, on="Station", how="left")

    merged = aq.merge(daily, on=["Station", "Date"], how="left",
                      validate="one_to_one", indicator=True)

    unmatched = int((merged["_merge"] != "both").sum())
    merged = merged.drop(columns="_merge")
    return merged, unmatched


def validate(merged, aq_rows):
    problems = []
    if len(merged) != aq_rows:
        problems.append(f"row count changed: {aq_rows} -> {len(merged)}")
    if merged.duplicated(["Station", "Date"]).any():
        problems.append("duplicate (Station, Date) after merge")

    weather_cols = ["temp_mean", "rh_mean", "wind_speed_mean", "pbl_mean",
                    "rain_sum", "solar_rad_sum", "pressure_mean"]
    for c in weather_cols:
        n = int(merged[c].isna().sum())
        if n:
            problems.append(f"{c}: {n} nulls after merge")

    # Physical sanity. These are wide bounds - they catch a unit mix-up or a
    # timezone error, not a genuinely unusual day.
    bounds = {"temp_mean": (5, 45), "rh_mean": (0, 100), "rh_max": (0, 100),
              "wind_speed_mean": (0, 30), "pressure_mean": (850, 1050),
              "rain_sum": (0, 500), "solar_rad_sum": (0, 40),
              "pbl_mean": (0, 5000), "wind_dir_resultant": (0, 360)}
    for c, (lo, hi) in bounds.items():
        if c in merged.columns:
            bad = merged[(merged[c] < lo) | (merged[c] > hi)]
            if len(bad):
                problems.append(
                    f"{c}: {len(bad)} values outside [{lo}, {hi}] "
                    f"(range {merged[c].min():.2f} to {merged[c].max():.2f})")
    return problems


def main():
    h = pd.read_csv(IN_HOURLY, parse_dates=["time"])
    provenance = pd.read_csv(IN_PROVENANCE)
    print(f"Read {IN_HOURLY}  {h.shape}")

    h = add_derived_hourly(h)
    h = assign_dates(h)
    daily = aggregate(h)
    daily = trim_and_check(daily)

    merged, unmatched = merge_with_aq(daily, provenance)
    if unmatched:
        print(f"\n[FAIL] {unmatched} air-quality rows found no weather match.")

    aq_rows = len(pd.read_csv(IN_AQ))
    problems = validate(merged, aq_rows)
    print("\nVALIDATION:", "all checks passed" if not problems else "")
    for p in problems:
        print("  [FAIL]", p)

    weather_only = ["Station", "Date"] + [
        c for c in daily.columns if c not in ("Station", "Date")]
    daily[weather_only].to_csv(OUT_WEATHER_DAILY, index=False)
    merged.to_csv(OUT_MERGED, index=False)

    print(f"\nWrote {OUT_WEATHER_DAILY}  {daily.shape}")
    print(f"Wrote {OUT_MERGED}  {merged.shape}")

    print("\nDaily weather summary (all stations pooled):")
    cols = ["temp_mean", "temp_range", "rh_mean", "dewpoint_mean",
            "rain_sum", "wind_speed_mean", "wind_steadiness",
            "pressure_mean", "pbl_mean", "ventilation_index",
            "solar_rad_sum", "cloud_cover_mean"]
    print(merged[cols].describe().T.round(2).to_string())

    print("\nHow much do the six stations actually differ? "
          "(std across stations on the same day, mean over all days)")
    spread = (merged.groupby("Date")[["temp_mean", "rh_mean",
                                      "wind_speed_mean", "pbl_mean"]]
              .std().mean().round(3))
    print(spread.to_string())
    print("Near-zero spread means the reanalysis grid cannot distinguish the\n"
          "stations and any per-station weather feature is largely nominal.")


if __name__ == "__main__":
    main()

Read Bengaluru_weather_hourly_raw.csv  (157824, 12)

[NOTE] 6 station-days built from fewer than 24 hours (expected: the final day loses its 24th accumulated hour to the timing shift):
            Station       Date  n_hours_instant  n_hours_accum
         BTM Layout 2026-06-30             24.0             23
             Hebbal 2026-06-30             24.0             23
Jayanagar 5th Block 2026-06-30             24.0             23
      Kasturi Nagar 2026-06-30             24.0             23
             Peenya 2026-06-30             24.0             23
         Silk Board 2026-06-30             24.0             23

VALIDATION: 
  [FAIL] pbl_mean: 1086 nulls after merge

Wrote Bengaluru_weather_daily.csv  (6576, 34)
Wrote Bengaluru_AQ_Weather_daily.csv  (6576, 68)

Daily weather summary (all stations pooled):
                    count     mean      std     min      25%      50%      75%       max
temp_mean          6576.0    23.72     2.35   17.74    22.20    23.28    24.84     31.6

In [6]:
"""
diagnose_pbl_missingness.py
===========================
Diagnoses WHY boundary_layer_height is missing, before anything is changed.
Writes no data files and modifies nothing.

    python diagnose_pbl_missingness.py

Two parts:

  PART A  (offline)  Characterises the missingness in the hourly file you
                     already have: is it by station, by date, by hour-of-day,
                     contiguous or scattered? Does any other variable share the
                     same gap? Is there a model-switch signature at the
                     boundary?

  PART B  (online)   A controlled A/B probe. Requests boundary_layer_height for
                     three short windows across several models and reports the
                     null fraction for each combination. Roughly 15 tiny
                     requests. This is what actually answers "would era5 fix
                     it" - the documentation does not state a coverage limit
                     for this variable, so the honest way to find out is to
                     ask the API directly.

Set RUN_ONLINE_PROBE = False to run Part A only.
"""

import pandas as pd
import numpy as np
import requests
import time

IN_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw.csv"
IN_MERGED = "../data/processed/Bengaluru_AQ_Weather_daily.csv"

RUN_ONLINE_PROBE = True
ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"

# One station is enough for the probe: the offline part establishes whether the
# gap is station-specific, and if it is not, a single coordinate answers it.
PROBE_LAT, PROBE_LON = 12.9135218, 77.5950804          # BTM Layout

PROBE_WINDOWS = [
    ("early  2023-08", "2023-08-01", "2023-08-07"),
    ("mid    2025-03", "2025-03-01", "2025-03-07"),
    ("late   2026-01", "2026-01-01", "2026-01-07"),
    ("latest 2026-06", "2026-06-24", "2026-06-30"),
]

PROBE_MODELS = ["best_match", "era5", "era5_seamless", "ecmwf_ifs", "era5_land"]

# Requested alongside PBL as a control: if temperature comes back but PBL does
# not, the gap is variable-specific, not a failed request.
PROBE_VARS = "boundary_layer_height,temperature_2m"


# ----------------------------------------------------------------------------
def part_a():
    print("=" * 74)
    print("PART A - characterising the gap in the data you already have")
    print("=" * 74)

    h = pd.read_csv(IN_HOURLY, parse_dates=["time"])
    h["date"] = h["time"].dt.normalize()
    pbl_null = h["boundary_layer_height"].isna()

    print(f"\nHourly file: {h.shape[0]:,} rows")
    print(f"PBL null hours: {pbl_null.sum():,} of {len(h):,} "
          f"({pbl_null.mean() * 100:.1f}%)")

    print("\n-- by station --")
    by_station = h.groupby("Station")["boundary_layer_height"].apply(
        lambda s: s.isna().mean() * 100).round(2)
    print(by_station.to_string())
    if by_station.nunique() == 1:
        print("  Identical across all six stations -> NOT station-specific,")
        print("  so not a coordinate or grid-cell problem.")
    else:
        print("  Differs by station -> investigate grid-cell selection.")

    print("\n-- by hour of day --")
    by_hour = h.groupby(h["time"].dt.hour)["boundary_layer_height"].apply(
        lambda s: s.isna().mean() * 100).round(1)
    print(by_hour.to_string())
    if by_hour.nunique() == 1:
        print("  Flat across all 24 hours -> whole days are missing, not")
        print("  particular hours. Rules out a sub-daily model (e.g. the")
        print("  6-hourly IFS Assimilation product) being the source.")
    else:
        print("  Varies by hour -> a coarser-timestep model may be supplying")
        print("  this variable, with gaps between its native timesteps.")

    print("\n-- by date --")
    daily_null = (h.groupby("date")["boundary_layer_height"]
                  .apply(lambda s: s.isna().all()))
    missing_dates = daily_null[daily_null].index
    if len(missing_dates) == 0:
        print("  No fully-missing days.")
    else:
        d = pd.Series(missing_dates)
        gaps = d.diff().dt.days.dropna()
        contiguous = bool((gaps == 1).all())
        print(f"  {len(missing_dates)} fully-missing days")
        print(f"  first: {d.min().date()}   last: {d.max().date()}")
        print(f"  contiguous block: {contiguous}")
        if not contiguous:
            breaks = d[1:][gaps.values > 1]
            print(f"  {len(breaks)} separate blocks; block starts: "
                  f"{[str(x.date()) for x in breaks[:10]]}")
        print(f"  station-days affected: {len(missing_dates) * h.Station.nunique()}")

    print("\n-- do other variables share the same gap? --")
    other = [c for c in h.columns
             if c not in ("Station", "time", "date", "boundary_layer_height")]
    if len(missing_dates):
        window = h[h["date"].isin(missing_dates)]
        share = window[other].isna().mean().mul(100).round(2)
        print(share.to_string())
        if (share == 0).all():
            print("  Every other variable is complete inside the gap.")
            print("  -> The request succeeded and the model returned data; only")
            print("     this one variable is absent. That is a variable-level")
            print("     coverage limit, not a failed or truncated download.")

    print("\n-- model-switch signature at the boundary --")
    # best_match stitches IFS/ERA5/ERA5-Land. A switch usually leaves a
    # footprint in an unrelated variable: a step in the mean or a change in
    # day-to-day variability. Surface pressure is a good probe because it is
    # elevation-sensitive and models differ in their terrain height.
    if len(missing_dates):
        boundary = pd.Timestamp(missing_dates.min())
        one = h[h["Station"] == h["Station"].iloc[0]].set_index("time")
        for var in ["surface_pressure", "temperature_2m", "wind_speed_10m"]:
            if var not in one.columns:
                continue
            before = one.loc[boundary - pd.Timedelta(days=60):boundary, var]
            after = one.loc[boundary:boundary + pd.Timedelta(days=60), var]
            print(f"  {var:20s} 60d before: mean={before.mean():8.2f} "
                  f"sd={before.std():6.2f} | 60d after: mean={after.mean():8.2f} "
                  f"sd={after.std():6.2f}")
        print("  A clear step in mean surface pressure across the boundary")
        print("  would indicate a model change (different terrain elevation).")
        print("  Similar values on both sides argue against a switch.")

    print("\n-- effect on the merged panel --")
    try:
        m = pd.read_csv(IN_MERGED, parse_dates=["Date"])
        m["year"] = m["Date"].dt.year
        tab = m.groupby("year").agg(
            rows=("Date", "size"),
            pbl_null=("pbl_mean", lambda s: int(s.isna().sum())),
            vent_null=("ventilation_index", lambda s: int(s.isna().sum())))
        tab["pbl_null_pct"] = (tab.pbl_null / tab.rows * 100).round(1)
        print(tab.to_string())

        if "CO_quality" in m.columns:
            co_bad = m["CO_quality"].str.startswith("unusable")
            pbl_bad = m["pbl_mean"].isna()
            both = int((co_bad & pbl_bad).sum())
            print(f"\n  Rows where CO is unusable AND PBL is missing: {both}")
            print(f"  CO unusable: {int(co_bad.sum())} | PBL missing: "
                  f"{int(pbl_bad.sum())}")
            if both > 0 and both == int(pbl_bad.sum()):
                print("  The two failures overlap completely. The same window")
                print("  loses a pollutant and the key mixing variable, and it")
                print("  is the period a chronological split uses for testing.")
    except FileNotFoundError:
        print(f"  ({IN_MERGED} not found, skipping)")


# ----------------------------------------------------------------------------
def probe(model, start, end):
    params = {"latitude": PROBE_LAT, "longitude": PROBE_LON,
              "start_date": start, "end_date": end,
              "hourly": PROBE_VARS, "timezone": "Asia/Kolkata",
              "models": model}
    try:
        r = requests.get(ENDPOINT, params=params, timeout=60)
        if r.status_code == 400:
            return {"status": "rejected", "reason": r.json().get("reason", "")[:60]}
        r.raise_for_status()
        p = r.json()["hourly"]
        if "boundary_layer_height" not in p:
            return {"status": "absent", "pbl_pct": None, "temp_pct": None}
        pbl = pd.Series(p["boundary_layer_height"])
        tmp = pd.Series(p.get("temperature_2m", []))
        return {"status": "ok",
                "pbl_pct": round(pbl.isna().mean() * 100, 1),
                "temp_pct": round(tmp.isna().mean() * 100, 1) if len(tmp) else None}
    except requests.RequestException as exc:
        return {"status": "error", "reason": str(exc)[:60]}


def part_b():
    print("\n" + "=" * 74)
    print("PART B - controlled probe: which model supplies PBL, and when")
    print("=" * 74)
    print("Values are % NULL for boundary_layer_height (temp in brackets as a")
    print("control). 0.0 means the variable is fully supplied.\n")

    rows = []
    for label, start, end in PROBE_WINDOWS:
        row = {"window": label}
        for model in PROBE_MODELS:
            res = probe(model, start, end)
            if res["status"] == "ok":
                row[model] = (f"{res['pbl_pct']}% ({res['temp_pct']}%)"
                              if res["pbl_pct"] is not None else "absent")
            elif res["status"] == "absent":
                row[model] = "not returned"
            else:
                row[model] = res["status"]
            time.sleep(0.8)
        rows.append(row)
        print(f"  {label} done")

    out = pd.DataFrame(rows).set_index("window")
    print("\n" + out.to_string())

    print("\nHow to read this:")
    print("  - A model showing 0.0% in every window is a drop-in replacement.")
    print("  - A model showing 0.0% early and 100% late has a coverage cutoff;")
    print("    switching to it would not help the recent window.")
    print("  - 'rejected' means that model string is not valid for this API.")
    print("  - If the temperature control is 0.0% while PBL is 100%, the")
    print("    request worked and the variable genuinely is not there.")


if __name__ == "__main__":
    part_a()
    if RUN_ONLINE_PROBE:
        part_b()
    print("\nNothing was written or modified.")

PART A - characterising the gap in the data you already have

Hourly file: 157,824 rows
PBL null hours: 26,208 of 157,824 (16.6%)

-- by station --
Station
BTM Layout             16.61
Hebbal                 16.61
Jayanagar 5th Block    16.61
Kasturi Nagar          16.61
Peenya                 16.61
Silk Board             16.61
  Identical across all six stations -> NOT station-specific,
  so not a coordinate or grid-cell problem.

-- by hour of day --
time
0     16.6
1     16.6
2     16.6
3     16.6
4     16.6
5     16.6
6     16.6
7     16.6
8     16.6
9     16.6
10    16.6
11    16.6
12    16.6
13    16.6
14    16.6
15    16.6
16    16.6
17    16.6
18    16.6
19    16.6
20    16.6
21    16.6
22    16.6
23    16.6
  Flat across all 24 hours -> whole days are missing, not
  particular hours. Rules out a sub-daily model (e.g. the
  6-hourly IFS Assimilation product) being the source.

-- by date --
  181 fully-missing days
  first: 2024-01-02   last: 2024-06-30
  contiguous block: True

In [7]:
"""
probe_pbl_coverage.py
=====================
Corrected probe. The previous one sampled four short windows chosen on a wrong
hypothesis and missed the actual gap entirely.

Instead of sampling, this requests boundary_layer_height for the ENTIRE study
window from each candidate model - one request per model, PBL only - and maps
coverage day by day. Three requests, complete answer, no sampling gaps.

Also re-runs the model-switch test in a way that is not confounded by season:
the previous version compared Nov-Dec against Jan-Feb, so the pressure step it
found was seasonal, not a model change. This compares the SAME calendar window
across three different years.

    python probe_pbl_coverage.py

Reads Bengaluru_weather_hourly_raw.csv. Writes nothing.
"""

import time

import numpy as np
import pandas as pd
import requests

ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"
IN_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw.csv"

START_DATE, END_DATE = "2023-07-01", "2026-06-30"
LAT, LON = 12.9135218, 77.5950804          # BTM Layout; gap is station-invariant
MODELS = ["best_match", "era5", "era5_seamless", "ecmwf_ifs"]

GAP_START, GAP_END = pd.Timestamp("2024-01-02"), pd.Timestamp("2024-06-30")


def fetch_full_window(model):
    params = {"latitude": LAT, "longitude": LON,
              "start_date": START_DATE, "end_date": END_DATE,
              "hourly": "boundary_layer_height,temperature_2m",
              "timezone": "Asia/Kolkata", "models": model}
    r = requests.get(ENDPOINT, params=params, timeout=120)
    if r.status_code == 400:
        return None, r.json().get("reason", "")[:80]
    r.raise_for_status()
    p = r.json()["hourly"]
    if "boundary_layer_height" not in p:
        return None, "variable not returned at all"
    df = pd.DataFrame({"time": pd.to_datetime(p["time"]),
                       "pbl": p["boundary_layer_height"],
                       "temp": p.get("temperature_2m")})
    return df, None


def coverage_blocks(daily_missing):
    """Contiguous runs of fully-missing days -> [(start, end, n_days), ...]."""
    idx = daily_missing[daily_missing].index
    if len(idx) == 0:
        return []
    s = pd.Series(idx)
    grp = (s.diff().dt.days.fillna(1) != 1).cumsum()
    return [(g.min().date(), g.max().date(), len(g)) for _, g in s.groupby(grp)]


def main():
    print("=" * 74)
    print("FULL-WINDOW PBL COVERAGE BY MODEL")
    print("=" * 74)
    print(f"{START_DATE} to {END_DATE}, one request per model, PBL only.\n")

    results = {}
    for model in MODELS:
        try:
            df, err = fetch_full_window(model)
        except requests.RequestException as exc:
            print(f"{model:16s} request failed: {exc}")
            continue
        if df is None:
            print(f"{model:16s} unavailable: {err}")
            continue

        df["date"] = df["time"].dt.normalize()
        daily_missing = df.groupby("date")["pbl"].apply(lambda s: s.isna().all())
        blocks = coverage_blocks(daily_missing)
        results[model] = (df, blocks)

        pct = df["pbl"].isna().mean() * 100
        tpct = df["temp"].isna().mean() * 100
        print(f"{model:16s} PBL null {pct:5.1f}%  (temperature control "
              f"{tpct:.1f}%)  missing blocks: {len(blocks)}")
        for a, b, n in blocks:
            print(f"{'':16s}   {a} -> {b}  ({n} days)")

        in_gap = df[(df["date"] >= GAP_START) & (df["date"] <= GAP_END)]
        gap_pct = in_gap["pbl"].isna().mean() * 100
        verdict = ("COVERS the gap" if gap_pct == 0 else
                   f"does NOT cover the gap ({gap_pct:.0f}% null there)")
        print(f"{'':16s}   -> {verdict}\n")
        time.sleep(1.0)

    # ---- agreement between models where both have data -----------------------
    if "best_match" in results and "era5" in results:
        bm = results["best_match"][0].set_index("time")["pbl"]
        e5 = results["era5"][0].set_index("time")["pbl"]
        both = pd.concat([bm.rename("best_match"), e5.rename("era5")],
                         axis=1).dropna()
        if len(both):
            r = both.corr().iloc[0, 1]
            bias = (both["era5"] - both["best_match"]).mean()
            print("-" * 74)
            print("AGREEMENT where both models supply PBL "
                  f"({len(both):,} hours):")
            print(f"  correlation           r = {r:.3f}")
            print(f"  mean difference       era5 - best_match = {bias:+.1f} m")
            print(f"  best_match mean       {both['best_match'].mean():.0f} m")
            print(f"  era5 mean             {both['era5'].mean():.0f} m")
            print("\n  High correlation and small bias mean a whole-series swap")
            print("  to era5 costs little. A large bias means the two are not")
            print("  interchangeable and patching only the gap would introduce")
            print("  a step change at the seam.")

    # ---- season-controlled model-switch test --------------------------------
    print("\n" + "=" * 74)
    print("MODEL-SWITCH TEST, SEASON CONTROLLED")
    print("=" * 74)
    print("Same calendar window (Jan 2 - Jun 30) in three different years.")
    print("If 2024 steps away from BOTH 2023 and 2025 in a variable that has")
    print("nothing to do with PBL, best_match served a different model there.\n")

    h = pd.read_csv(IN_HOURLY, parse_dates=["time"])
    h = h[h["Station"] == "BTM Layout"].copy()
    h["year"] = h["time"].dt.year
    mask = ((h["time"].dt.month * 100 + h["time"].dt.day >= 102) &
            (h["time"].dt.month * 100 + h["time"].dt.day <= 630))
    sub = h[mask]

    for var in ["surface_pressure", "temperature_2m", "wind_speed_10m",
                "dew_point_2m"]:
        if var not in sub.columns:
            continue
        stats = sub.groupby("year")[var].agg(["mean", "std"]).round(2)
        line = "  ".join(f"{y}: {row['mean']:8.2f} (sd {row['std']:5.2f})"
                         for y, row in stats.iterrows())
        print(f"  {var:20s} {line}")

    print("\n  A 2024 value sitting between 2023 and 2025 is seasonal variation.")
    print("  A 2024 value offset from both, especially in surface pressure")
    print("  (which is elevation-sensitive and differs between model grids),")
    print("  indicates the model actually changed.")


if __name__ == "__main__":
    main()

FULL-WINDOW PBL COVERAGE BY MODEL
2023-07-01 to 2026-06-30, one request per model, PBL only.

best_match       PBL null  16.6%  (temperature control 0.0%)  missing blocks: 1
                   2024-01-02 -> 2024-06-30  (181 days)
                   -> does NOT cover the gap (100% null there)

era5             PBL null  16.6%  (temperature control 0.0%)  missing blocks: 1
                   2024-01-02 -> 2024-06-30  (181 days)
                   -> does NOT cover the gap (100% null there)

era5_seamless    PBL null  16.6%  (temperature control 0.0%)  missing blocks: 1
                   2024-01-02 -> 2024-06-30  (181 days)
                   -> does NOT cover the gap (100% null there)

ecmwf_ifs        PBL null  40.9%  (temperature control 0.0%)  missing blocks: 2
                   2023-07-01 -> 2024-08-31  (428 days)
                   2025-09-19 -> 2025-10-07  (19 days)
                   -> does NOT cover the gap (100% null there)

---------------------------------------------------

In [8]:
"""
refetch_pbl_single_model.py
===========================
RUN ONLY IF probe_pbl_coverage.py confirms that a single model supplies
boundary_layer_height across the whole 2023-07-01 to 2026-06-30 window.

Re-fetches PBL alone, from that one model, for all six stations, and writes a
new hourly file with the PBL column replaced. Everything else is carried over
untouched from the existing raw hourly file.

    python refetch_pbl_single_model.py

INPUT   Bengaluru_weather_hourly_raw.csv          (not modified)
OUTPUT  Bengaluru_weather_hourly_raw_v2.csv
        pbl_swap_comparison.csv

THEN    edit aggregate_and_merge_weather.py:
            IN_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw_v2.csv"
        and re-run it.

WHY REPLACE THE WHOLE COLUMN RATHER THAN PATCH THE GAP
------------------------------------------------------
Patching only 2024-01-02 to 2024-06-30 would leave one column built from two
different models with two different grid resolutions, joined at two seams. Any
systematic offset between them becomes a step change in the PBL series - and
therefore in the ventilation index - at exactly those two dates. A model would
happily learn that step as if it were atmospheric.

Replacing the entire column costs six more requests and guarantees the series
is internally consistent. The trade-off is that PBL may then come from a
coarser grid than the other variables. That is a documentable limitation;
a fabricated discontinuity is not.

The script quantifies the difference between old and new PBL over the period
where both exist, so the cost of the swap is measured rather than assumed.
"""

import sys
import time

import numpy as np
import pandas as pd
import requests

ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"

# Set this to whichever model probe_pbl_coverage.py showed covering the FULL
# window with zero missing blocks.
MODEL = "era5"

START_DATE, END_DATE = "2023-07-01", "2026-06-30"
TIMEZONE = "Asia/Kolkata"

IN_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw.csv"
OUT_HOURLY = "../data/processed/Bengaluru_weather_hourly_raw_v2.csv"
OUT_COMPARISON = "../data/processed/pbl_swap_comparison.csv"

STATIONS = {
    "BTM Layout": (12.9135218, 77.5950804),
    "Hebbal": (13.029152, 77.585901),
    "Jayanagar 5th Block": (12.920984, 77.584908),
    "Kasturi Nagar": (13.004533, 77.664561),
    "Peenya": (13.0270199, 77.494094),
    "Silk Board": (12.917348, 77.622813),
}


def fetch_pbl(name, lat, lon):
    params = {"latitude": lat, "longitude": lon,
              "start_date": START_DATE, "end_date": END_DATE,
              "hourly": "boundary_layer_height",
              "timezone": TIMEZONE, "models": MODEL}
    for attempt in range(1, 5):
        try:
            r = requests.get(ENDPOINT, params=params, timeout=120)
            if r.status_code == 400:
                raise SystemExit(f"[FATAL] {name}: "
                                 f"{r.json().get('reason', r.text)}")
            r.raise_for_status()
            p = r.json()
            return pd.DataFrame({
                "Station": name,
                "time": pd.to_datetime(p["hourly"]["time"]),
                "pbl_new": p["hourly"]["boundary_layer_height"],
            }), p
        except requests.RequestException as exc:
            print(f"  attempt {attempt} failed ({exc}); retrying")
            time.sleep(2 ** attempt)
    raise SystemExit(f"[FATAL] {name}: all retries failed")


def main():
    old = pd.read_csv(IN_HOURLY, parse_dates=["time"])
    print(f"Read {IN_HOURLY}  {old.shape}")
    print(f"Re-fetching boundary_layer_height from models={MODEL}\n")

    frames = []
    for name, (lat, lon) in STATIONS.items():
        print(f"  {name} ...")
        df, payload = fetch_pbl(name, lat, lon)
        nulls = int(df["pbl_new"].isna().sum())
        print(f"    {len(df):,} hours, {nulls} null, grid "
              f"({payload['latitude']:.4f}, {payload['longitude']:.4f}), "
              f"elevation {payload['elevation']} m")
        if nulls:
            print(f"    [WARN] {MODEL} still returns nulls. Do not proceed "
                  f"until probe_pbl_coverage.py shows a clean model.")
        frames.append(df)
        time.sleep(1.0)

    new = pd.concat(frames, ignore_index=True)

    merged = old.merge(new, on=["Station", "time"], how="left",
                       validate="one_to_one")
    if len(merged) != len(old):
        sys.exit(f"[FATAL] row count changed {len(old)} -> {len(merged)}")

    # ---- measure what the swap costs, where both exist ----------------------
    both = merged.dropna(subset=["boundary_layer_height", "pbl_new"])
    comparison = None
    if len(both):
        diff = both["pbl_new"] - both["boundary_layer_height"]
        r = both[["boundary_layer_height", "pbl_new"]].corr().iloc[0, 1]
        print(f"\nOverlap where both models supply PBL: {len(both):,} hours")
        print(f"  correlation                r     = {r:.3f}")
        print(f"  mean difference (new-old)        = {diff.mean():+.1f} m")
        print(f"  median absolute difference       = {diff.abs().median():.1f} m")
        print(f"  old mean {both['boundary_layer_height'].mean():.0f} m | "
              f"new mean {both['pbl_new'].mean():.0f} m")
        comparison = (both.assign(diff=diff)
                      .groupby(both["time"].dt.to_period("M"))
                      .agg(n=("diff", "size"),
                           old_mean=("boundary_layer_height", "mean"),
                           new_mean=("pbl_new", "mean"),
                           mean_diff=("diff", "mean"))
                      .round(1).reset_index())
        comparison.to_csv(OUT_COMPARISON, index=False)
        print(f"\n  Monthly comparison written to {OUT_COMPARISON} - check it")
        print("  for a jump in mean_diff around 2024-07, which would indicate")
        print("  best_match changed model there.")

    # ---- swap ---------------------------------------------------------------
    merged["boundary_layer_height_bestmatch"] = merged["boundary_layer_height"]
    merged["boundary_layer_height"] = merged["pbl_new"]
    merged["pbl_model"] = MODEL
    merged = merged.drop(columns="pbl_new")

    remaining = int(merged["boundary_layer_height"].isna().sum())
    print(f"\nPBL nulls after swap: {remaining} "
          f"(was {int(old['boundary_layer_height'].isna().sum()):,})")

    merged.to_csv(OUT_HOURLY, index=False)
    print(f"Wrote {OUT_HOURLY}  {merged.shape}")
    print("\nThe original best_match values are kept as")
    print("'boundary_layer_height_bestmatch' so the swap stays auditable.")
    print(f"\nNext: set IN_HOURLY = \"{OUT_HOURLY}\" in "
          f"aggregate_and_merge_weather.py and re-run it.")


if __name__ == "__main__":
    main()

Read Bengaluru_weather_hourly_raw.csv  (157824, 12)
Re-fetching boundary_layer_height from models=era5

  BTM Layout ...
    26,304 hours, 4368 null, grid (13.0000, 77.5000), elevation 915.0 m
    [WARN] era5 still returns nulls. Do not proceed until probe_pbl_coverage.py shows a clean model.
  Hebbal ...
    26,304 hours, 4368 null, grid (13.0000, 77.5000), elevation 921.0 m
    [WARN] era5 still returns nulls. Do not proceed until probe_pbl_coverage.py shows a clean model.
  Jayanagar 5th Block ...
    26,304 hours, 4368 null, grid (13.0000, 77.5000), elevation 915.0 m
    [WARN] era5 still returns nulls. Do not proceed until probe_pbl_coverage.py shows a clean model.
  Kasturi Nagar ...
    26,304 hours, 4368 null, grid (13.0000, 77.7500), elevation 905.0 m
    [WARN] era5 still returns nulls. Do not proceed until probe_pbl_coverage.py shows a clean model.
  Peenya ...
    26,304 hours, 4368 null, grid (13.0000, 77.5000), elevation 898.0 m
    [WARN] era5 still returns nulls. Do not

In [9]:
"""
build_modelling_dataset_no_pbl.py
=================================
Creates the modelling dataset with boundary-layer variables removed.

    python build_modelling_dataset_no_pbl.py

INPUTS  (all read-only, none modified)
    Bengaluru_AQ_Weather_daily.csv        the v1 merged panel
    Bengaluru_AirQuality_repaired.csv     to verify the AQ block is untouched

OUTPUTS
    Bengaluru_AQ_Weather_daily_v3_noPBL.csv    the modelling dataset
    column_manifest_v3.csv                     every column, kept or dropped

WHAT IS REMOVED AND WHY
-----------------------
boundary_layer_height is absent from every available reanalysis model for
2024-01-02 to 2024-06-30 (181 days, all six stations, 16.6% of the series).
Neither best_match nor era5 supplies it. The variable is dropped rather than
imputed: reconstructing six months of mixing height from its own neighbours
would place an invented series underneath the ventilation index, and the model
would learn it as though it were atmospheric.

    pbl_mean, pbl_min, pbl_max      dropped from the daily panel
    ventilation_index               dropped (it is PBL x wind speed, so it
                                    inherits the gap exactly)

boundary_layer_height itself is an HOURLY column and lives only in
Bengaluru_weather_hourly_raw.csv, which is deliberately left untouched. The
raw hourly file remains the complete record of what the API returned; the
modelling dataset is where the variable is absent. That separation is what
makes the decision reversible if a model later gains coverage.

WHAT THIS SCRIPT DOES NOT DO
----------------------------
No imputation. No feature engineering. No AQI computation. No re-fetching.
It removes four columns and then tries hard to prove nothing else changed.
"""

import sys

import numpy as np
import pandas as pd

IN_MERGED = "../data/processed/Bengaluru_AQ_Weather_daily.csv"
IN_AQ = "../data/processed/Bengaluru_AirQuality_repaired.csv"

OUT_DATA = "../data/processed/Bengaluru_AQ_Weather_daily_v3_noPBL.csv"
OUT_MANIFEST = "../data/processed/column_manifest_v3.csv"

DROP = ["pbl_mean", "pbl_min", "pbl_max", "ventilation_index"]

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

STUDY_START = pd.Timestamp("2023-07-01")
STUDY_END = pd.Timestamp("2026-06-30")
EXPECTED_ROWS = 6576
EXPECTED_DAYS = 1096
EXPECTED_STATIONS = 6

# Wide enough to catch a unit error or timezone fault, not an unusual day.
BOUNDS = {
    "temp_mean": (5, 45), "temp_min": (2, 40), "temp_max": (10, 50),
    "temp_range": (0, 30),
    "rh_mean": (0, 100), "rh_min": (0, 100), "rh_max": (0, 100),
    "dewpoint_mean": (-5, 30),
    "pressure_mean": (850, 1050), "pressure_range": (0, 30),
    "cloud_cover_mean": (0, 100),
    "wind_speed_mean": (0, 30), "wind_speed_max": (0, 50),
    "wind_speed_resultant": (0, 30), "wind_dir_resultant": (0, 360),
    "wind_steadiness": (0, 1.001),
    "rain_sum": (0, 500), "rain_hours": (0, 24),
    "solar_rad_sum": (0, 40),
}


# ----------------------------------------------------------------------------
def check_structure(df, problems):
    if len(df) != EXPECTED_ROWS:
        problems.append(f"row count {len(df)}, expected {EXPECTED_ROWS}")
    if df["Station"].nunique() != EXPECTED_STATIONS:
        problems.append(f"{df['Station'].nunique()} stations, "
                        f"expected {EXPECTED_STATIONS}")
    if df.duplicated(["Station", "Date"]).any():
        problems.append("duplicate (Station, Date) pairs")

    for station, g in df.groupby("Station"):
        d = g["Date"].sort_values()
        if len(d) != EXPECTED_DAYS:
            problems.append(f"{station}: {len(d)} days, expected {EXPECTED_DAYS}")
        if d.min() != STUDY_START or d.max() != STUDY_END:
            problems.append(f"{station}: range {d.min().date()} to "
                            f"{d.max().date()}")
        gaps = set(d.diff().dropna().dt.days.unique())
        if gaps != {1}:
            problems.append(f"{station}: non-daily spacing {gaps}")


def check_drop_was_exact(before, after, problems):
    """Exactly the intended columns went, and nothing else moved."""
    removed = [c for c in before.columns if c not in after.columns]
    added = [c for c in after.columns if c not in before.columns]

    if sorted(removed) != sorted(DROP):
        problems.append(f"removed {sorted(removed)}, intended {sorted(DROP)}")
    if added:
        problems.append(f"unexpected new columns: {added}")

    # Every retained column must be value-for-value identical to v1.
    changed = []
    for c in after.columns:
        a, b = before[c], after[c]
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
            same = np.allclose(a.astype(float), b.astype(float),
                               equal_nan=True)
        else:
            same = a.astype(str).equals(b.astype(str))
        if not same:
            changed.append(c)
    if changed:
        problems.append(f"retained columns whose values changed: {changed}")


def check_aq_block_untouched(after, problems):
    """The pollutant half must be byte-identical to the repair-pass output."""
    aq = pd.read_csv(IN_AQ, parse_dates=["Date"])
    key = ["Station", "Date"]
    merged = aq.merge(after, on=key, how="inner", suffixes=("_orig", ""))
    if len(merged) != len(aq):
        problems.append(f"AQ join produced {len(merged)} rows, expected {len(aq)}")
        return

    for c in aq.columns:
        if c in key:
            continue
        orig, now = merged[f"{c}_orig"], merged[c]
        if pd.api.types.is_numeric_dtype(orig):
            same = np.allclose(orig.astype(float), now.astype(float),
                               equal_nan=True)
        else:
            same = orig.astype(str).equals(now.astype(str))
        if not same:
            problems.append(f"AQ column altered: {c}")


def check_completeness(df, problems):
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        for c, n in nulls.items():
            problems.append(f"{c}: {n} nulls remain")


def check_bounds(df, problems):
    for c, (lo, hi) in BOUNDS.items():
        if c not in df.columns:
            continue
        bad = df[(df[c] < lo) | (df[c] > hi)]
        if len(bad):
            problems.append(f"{c}: {len(bad)} values outside [{lo}, {hi}] "
                            f"(observed {df[c].min():.2f} to {df[c].max():.2f})")


def check_no_pbl_residue(df, problems):
    residue = [c for c in df.columns
               if any(t in c.lower() for t in ("pbl", "boundary_layer",
                                               "ventilation"))]
    if residue:
        problems.append(f"PBL-derived columns still present: {residue}")


# ----------------------------------------------------------------------------
def report_model_seam(df):
    """The PBL gap proves best_match's model selection changes mid-window.
    If it does, the RETAINED variables may carry a step at the same dates.
    Same calendar window, three years, so season is held constant."""
    print("\n" + "=" * 74)
    print("MODEL-SEAM CHECK ON THE RETAINED WEATHER VARIABLES")
    print("=" * 74)
    print("Jan 2 - Jun 30 in each year. The PBL gap covers exactly this window")
    print("in 2024. If best_match served a different model there, 2024 should")
    print("sit OFF-TREND relative to both 2023 and 2025, not between them.\n")

    d = df.copy()
    d["year"] = d["Date"].dt.year
    mmdd = d["Date"].dt.month * 100 + d["Date"].dt.day
    sub = d[(mmdd >= 102) & (mmdd <= 630)]

    cols = ["temp_mean", "rh_mean", "dewpoint_mean", "pressure_mean",
            "wind_speed_mean", "cloud_cover_mean", "solar_rad_sum"]
    cols = [c for c in cols if c in sub.columns]

    stats = sub.groupby("year")[cols].mean().round(2)
    print(stats.to_string())

    # How far 2024 sits from the other years, measured in units of the
    # variable's OWN day-to-day variability. Scaling by the gap between
    # neighbouring years would be wrong: when those years happen to be close,
    # the denominator collapses and any deviation looks significant. A model
    # change large enough to matter must move the mean by an appreciable
    # fraction of a daily standard deviation.
    SEAM_THRESHOLD = 0.30       # in daily standard deviations
    print(f"\nDeviation of 2024 from the other years, in units of each")
    print(f"variable's daily standard deviation (flagged above "
          f"{SEAM_THRESHOLD}):\n")

    daily_sd = sub[cols].std()
    flagged = []
    for c in cols:
        others = [y for y in stats.index if y != 2024]
        if not others:
            continue
        v24 = stats.loc[2024, c]
        baseline = stats.loc[others, c].median()
        sd = daily_sd[c] if daily_sd[c] > 0 else np.nan
        dev = abs(v24 - baseline) / sd if sd and not np.isnan(sd) else np.nan
        if not np.isnan(dev) and dev > SEAM_THRESHOLD:
            flagged.append(c)
            verdict = "OFF-TREND - possible model change"
        else:
            verdict = "consistent with the other years"
        print(f"  {c:20s} 2024={v24:8.2f}  other years={baseline:8.2f}  "
              f"dev={dev:5.2f} sd  {verdict}")

    if flagged:
        print(f"\n  [ATTENTION] {flagged} sit off-trend in the gap window.")
        print("  Surface pressure is the most diagnostic of these, because")
        print("  model grids differ in terrain elevation. If pressure is")
        print("  flagged, treat a model switch as likely and record it as a")
        print("  known inhomogeneity in the weather series for H1 2024.")
    else:
        print("\n  No retained variable sits off-trend in the gap window.")
        print("  The PBL absence therefore looks like a variable-level")
        print("  coverage limit rather than a wholesale model change, and")
        print("  the retained weather series can be treated as homogeneous.")
        print("  This is a weaker test than a direct model label, so state it")
        print("  as consistency evidence, not proof.")


def report_aqi_readiness(df):
    """Dropping PBL cannot affect AQI: the CPCB index is a function of
    pollutant sub-indices only. Shown here rather than asserted."""
    print("\n" + "=" * 74)
    print("AQI COMPUTABILITY (unaffected by the PBL decision)")
    print("=" * 74)

    qcols = [f"{p}_quality" for p in POLLUTANTS]
    if not all(c in df.columns for c in qcols):
        print("  Quality columns not found; skipping.")
        return

    usable = pd.DataFrame(
        {p: ~df[f"{p}_quality"].str.startswith("unusable")
         for p in POLLUTANTS})
    n_usable = usable.sum(axis=1)
    has_pm = usable["PM2.5"] | usable["PM10"]

    # CPCB: at least three pollutants, one of which must be PM2.5 or PM10.
    computable = (n_usable >= 3) & has_pm

    print(f"  Rows where CPCB AQI is computable: {int(computable.sum()):,} "
          f"of {len(df):,} ({computable.mean() * 100:.1f}%)")
    print("\n  Distribution of usable pollutants per station-day:")
    print(n_usable.value_counts().sort_index().to_string())
    print("\n  By station:")
    print(df.assign(ok=computable, n=n_usable)
          .groupby("Station")
          .agg(aqi_computable_pct=("ok", lambda s: round(s.mean() * 100, 1)),
               mean_pollutants=("n", lambda s: round(s.mean(), 2)))
          .to_string())

    print("\n  The CPCB sub-index formula takes pollutant concentrations only.")
    print("  No meteorological term appears in it, so removing PBL changes")
    print("  none of the above. Weather enters your work solely as a predictor")
    print("  of AQI, never as an input to its definition.")
    print("\n  Carry the usable-pollutant count as a column when you compute")
    print("  AQI: an index built from three pollutants is not comparable to")
    print("  one built from six.")


# ----------------------------------------------------------------------------
def main():
    before = pd.read_csv(IN_MERGED, parse_dates=["Date"])
    print(f"Read {IN_MERGED}  {before.shape}")

    missing = [c for c in DROP if c not in before.columns]
    if missing:
        sys.exit(f"[FATAL] expected columns not present: {missing}")

    after = before.drop(columns=DROP).copy()

    problems = []
    check_structure(after, problems)
    check_drop_was_exact(before, after, problems)
    check_aq_block_untouched(after, problems)
    check_completeness(after, problems)
    check_bounds(after, problems)
    check_no_pbl_residue(after, problems)

    print("\n" + "=" * 74)
    print("VALIDATION")
    print("=" * 74)
    if problems:
        for p in problems:
            print("  [FAIL]", p)
        print("\nNothing written. Fix the above and re-run.")
        sys.exit(1)
    print("  [PASS] 6,576 rows, 6 stations x 1,096 consecutive days")
    print("  [PASS] exactly the 4 intended columns removed, no others")
    print("  [PASS] every retained column identical to v1, value for value")
    print("  [PASS] air-quality block identical to the repair-pass output")
    print("  [PASS] zero nulls in every remaining column")
    print("  [PASS] all weather variables within physical bounds")
    print("  [PASS] no PBL, boundary-layer or ventilation columns remain")

    manifest = pd.DataFrame({
        "column": list(before.columns),
        "status": ["dropped" if c in DROP else "kept" for c in before.columns],
        "dtype": [str(before[c].dtype) for c in before.columns],
        "nulls_in_v1": [int(before[c].isna().sum()) for c in before.columns],
    })
    manifest.to_csv(OUT_MANIFEST, index=False)
    after.to_csv(OUT_DATA, index=False)

    print(f"\nWrote {OUT_DATA}  {after.shape}")
    print(f"Wrote {OUT_MANIFEST}  ({len(manifest)} columns documented)")
    print(f"\nUnchanged on disk: {IN_MERGED}, {IN_AQ}, "
          f"Bengaluru_weather_hourly_raw.csv and any v2 file.")

    report_model_seam(after)
    report_aqi_readiness(after)


if __name__ == "__main__":
    main()

Read Bengaluru_AQ_Weather_daily.csv  (6576, 68)

VALIDATION
  [PASS] 6,576 rows, 6 stations x 1,096 consecutive days
  [PASS] exactly the 4 intended columns removed, no others
  [PASS] every retained column identical to v1, value for value
  [PASS] air-quality block identical to the repair-pass output
  [PASS] zero nulls in every remaining column
  [PASS] all weather variables within physical bounds
  [PASS] no PBL, boundary-layer or ventilation columns remain

Wrote Bengaluru_AQ_Weather_daily_v3_noPBL.csv  (6576, 64)
Wrote column_manifest_v3.csv  (68 columns documented)

Unchanged on disk: Bengaluru_AQ_Weather_daily.csv, Bengaluru_AirQuality_repaired.csv, Bengaluru_weather_hourly_raw.csv and any v2 file.

MODEL-SEAM CHECK ON THE RETAINED WEATHER VARIABLES
Jan 2 - Jun 30 in each year. The PBL gap covers exactly this window
in 2024. If best_match served a different model there, 2024 should
sit OFF-TREND relative to both 2023 and 2025, not between them.

      temp_mean  rh_mean  dewpoin

In [16]:

"""
eda_step7.py
============
Exploratory analysis of the merged AQ + weather panel. No modelling, no AQI
computation, no feature engineering.

    pip install pandas numpy scipy statsmodels matplotlib
    python eda_step7.py

INPUT   Bengaluru_AQ_Weather_daily_v3_noPBL.csv   (read-only)
OUTPUT  eda_outputs/   figures (.png) and tables (.csv)

TWO METHODOLOGICAL CHOICES THAT SHAPE EVERYTHING BELOW
------------------------------------------------------
1. MASKING. Every statistic uses usable cells only. A cell flagged
   `unusable:*` by the repair pass carries a number, but that number is a
   stuck sensor, a regime-shifted analyser or a network failure. Including it
   would put roughly 18% of NO2 and 22% of CO into every correlation and
   distribution in this file. Masked cells are excluded, never imputed.

2. DESEASONALISATION. Raw correlations between a pollutant and a weather
   variable in a monsoon climate are dominated by shared seasonality:
   rainfall and PM2.5 both follow the monsoon, so they correlate strongly
   whether or not rain is doing anything to PM2.5 on any given day. Every
   correlation is therefore reported twice - raw, and after removing the
   station-by-calendar-month mean from both series. The second number is the
   one that carries information about day-to-day covariation.

Neither number is evidence of causation. Both pollutants and weather respond
to the same synoptic conditions, and the direction of any relationship here is
an assumption imported from atmospheric physics, not something these data can
establish.
"""

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy import stats as sps
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")

IN_DATA = "../data/processed/Bengaluru_AQ_Weather_daily_v3_noPBL.csv"
OUT_DIR = "../eda_outputs"

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

UNITS = {
    "PM2.5": "µg/m³",
    "PM10": "µg/m³",
    "NO2": "µg/m³",
    "SO2": "µg/m³",
    "CO": "mg/m³",
    "Ozone": "µg/m³",
}

WEATHER = [
    "temp_mean",
    "temp_max",
    "temp_min",
    "temp_range",
    "rh_mean",
    "rh_min",
    "dewpoint_mean",
    "pressure_mean",
    "pressure_range",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_speed_resultant",
    "wind_steadiness",
    "rain_sum",
    "rain_hours",
    "solar_rad_sum",
    "cloud_cover_mean",
]

# India Meteorological Department season definitions.
SEASONS = {
    1: "Winter",
    2: "Winter",
    3: "Pre-monsoon",
    4: "Pre-monsoon",
    5: "Pre-monsoon",
    6: "Monsoon",
    7: "Monsoon",
    8: "Monsoon",
    9: "Monsoon",
    10: "Post-monsoon",
    11: "Post-monsoon",
    12: "Winter",
}

MIN_BLOCK = 365          # shortest contiguous run accepted for a unit-root test


# ----------------------------------------------------------------------------
def load():
    df = pd.read_csv(IN_DATA, parse_dates=["Date"])

    df["month"] = df["Date"].dt.month
    df["year"] = df["Date"].dt.year
    df["season"] = df["month"].map(SEASONS)

    # Masked copy: unusable cells become NaN for analysis only.
    for p in POLLUTANTS:
        bad = df[f"{p}_quality"].str.startswith("unusable")
        df[f"{p}_use"] = df[p].where(~bad)

    return df


def deseasonalise(df, cols):
    """
    Remove the station-by-calendar-month mean from each column.

    Leaves day-to-day anomalies, which is what a same-day correlation should
    be measuring once shared seasonality is taken out.
    """
    out = pd.DataFrame(index=df.index)

    for c in cols:
        grp = df.groupby(["Station", "month"])[c]
        out[c] = df[c] - grp.transform("mean")

    return out


# ---------------------------------------------------------------- 1. coverage
def section_missingness(df):
    print("\n" + "=" * 74)
    print("1. DATA AVAILABILITY AFTER MASKING")
    print("=" * 74)

    rows = []

    for p in POLLUTANTS:
        q = df[f"{p}_quality"]

        rows.append({
            "pollutant": p,
            "measured_%": round(
                q.isin(
                    ["measured", "flag:pm_ratio_marginal"]
                ).mean() * 100,
                1,
            ),
            "estimated_%": round(
                q.str.startswith(("imputed", "repair")).mean() * 100,
                1,
            ),
            "unusable_%": round(
                q.str.startswith("unusable").mean() * 100,
                1,
            ),
            "n_usable": int(df[f"{p}_use"].notna().sum()),
        })

    tab = pd.DataFrame(rows)

    print(tab.to_string(index=False))
    tab.to_csv(f"{OUT_DIR}/01_availability.csv", index=False)

    print("\nUsable station-days by station x pollutant:")

    per = pd.DataFrame({
        p: df.groupby("Station")[f"{p}_use"].apply(
            lambda s: int(s.notna().sum())
        )
        for p in POLLUTANTS
    })

    print(per.to_string())
    per.to_csv(f"{OUT_DIR}/01_usable_by_station.csv")

    print("\nUsable share by year (%):")

    yr = pd.DataFrame({
        p: df.groupby("year")[f"{p}_use"].apply(
            lambda s: round(s.notna().mean() * 100, 1)
        )
        for p in POLLUTANTS
    })

    print(yr.to_string())

    print("\n  Weather variables have no missing values by construction:")
    print(
        f"  reanalysis is spatially and temporally complete. "
        f"Nulls in weather columns: "
        f"{int(df[WEATHER].isna().sum().sum())}"
    )


# ------------------------------------------------------------- 2. descriptives
def section_descriptives(df):
    print("\n" + "=" * 74)
    print("2. DESCRIPTIVE STATISTICS (usable cells only)")
    print("=" * 74)

    rows = []

    for p in POLLUTANTS:
        s = df[f"{p}_use"].dropna()

        rows.append({
            "variable": f"{p} ({UNITS[p]})",
            "n": len(s),
            "mean": s.mean(),
            "sd": s.std(),
            "min": s.min(),
            "p25": s.quantile(.25),
            "median": s.median(),
            "p75": s.quantile(.75),
            "p95": s.quantile(.95),
            "max": s.max(),
            "skew": s.skew(),
            "kurtosis": s.kurtosis(),
        })

    pol = pd.DataFrame(rows).round(2)

    print(pol.to_string(index=False))
    pol.to_csv(
        f"{OUT_DIR}/02_descriptives_pollutants.csv",
        index=False,
    )

    wx = df[WEATHER].describe().T[
        ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    ].round(2)

    wx["skew"] = df[WEATHER].skew().round(2)

    print("\nWeather:")
    print(wx.to_string())
    wx.to_csv(f"{OUT_DIR}/02_descriptives_weather.csv")

    print(
        "\n  Positive skew in the pollutants is expected and matters: it means"
    )
    print(
        "  Pearson correlations are leverage-sensitive, which is why Spearman"
    )
    print("  is reported alongside throughout.")


# -------------------------------------------------------------- 3. correlation
def section_correlation(df):
    print("\n" + "=" * 74)
    print("3. POLLUTANT-WEATHER CORRELATION, RAW vs DESEASONALISED")
    print("=" * 74)

    pol_cols = [f"{p}_use" for p in POLLUTANTS]

    des = deseasonalise(
        df,
        pol_cols + WEATHER
    )

    raw_s = pd.DataFrame(
        index=POLLUTANTS,
        columns=WEATHER,
        dtype=float,
    )

    des_s = raw_s.copy()

    for p in POLLUTANTS:
        for w in WEATHER:

            a = df[[f"{p}_use", w]].dropna()

            if len(a) > 30:
                raw_s.loc[p, w] = sps.spearmanr(
                    a.iloc[:, 0],
                    a.iloc[:, 1]
                )[0]

            b = des[[f"{p}_use", w]].dropna()

            if len(b) > 30:
                des_s.loc[p, w] = sps.spearmanr(
                    b.iloc[:, 0],
                    b.iloc[:, 1]
                )[0]

    print("\nSpearman rho, RAW (contains shared seasonality):")
    print(raw_s.round(2).to_string())

    print("\nSpearman rho, DESEASONALISED (day-to-day anomalies):")
    print(des_s.round(2).to_string())

    raw_s.round(4).to_csv(
        f"{OUT_DIR}/03_corr_raw_spearman.csv"
    )

    des_s.round(4).to_csv(
        f"{OUT_DIR}/03_corr_deseasonalised_spearman.csv"
    )

    shrink = (
        raw_s.abs() - des_s.abs()
    ).round(2)

    print(
        "\nHow much of each correlation was seasonality "
        "(|raw| - |deseasonalised|):"
    )

    print(shrink.to_string())

    shrink.to_csv(
        f"{OUT_DIR}/03_corr_seasonal_share.csv"
    )

    print(
        "\n  Large positive values mean the raw association was mostly two"
    )
    print(
        "  series sharing a monsoon cycle, not a day-to-day relationship."
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(18, 5)
    )

    for ax, mat, title in zip(
        axes,
        [raw_s, des_s],
        ["Raw Spearman", "Deseasonalised Spearman"]
    ):

        im = ax.imshow(
            mat.astype(float),
            cmap="RdBu_r",
            vmin=-.6,
            vmax=.6,
        )

        ax.set_xticks(range(len(WEATHER)))
        ax.set_xticklabels(
            WEATHER,
            rotation=90,
            fontsize=8
        )

        ax.set_yticks(range(len(POLLUTANTS)))
        ax.set_yticklabels(POLLUTANTS)

        ax.set_title(title)

        for i in range(len(POLLUTANTS)):
            for j in range(len(WEATHER)):

                v = mat.iloc[i, j]

                if pd.notna(v):
                    ax.text(
                        j,
                        i,
                        f"{v:.2f}",
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color="white" if abs(v) > .35 else "black",
                    )

        fig.colorbar(
            im,
            ax=ax,
            fraction=.02
        )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/03_correlation_heatmaps.png",
        dpi=140,
    )

    plt.close()

    # Weather-weather collinearity, for the modelling step later.
    wc = df[WEATHER].corr(method="spearman")

    wc.round(3).to_csv(
        f"{OUT_DIR}/03_weather_collinearity.csv"
    )

    high = [
        (a, b, round(wc.loc[a, b], 2))
        for i, a in enumerate(WEATHER)
        for b in WEATHER[i + 1:]
        if abs(wc.loc[a, b]) > 0.85
    ]

    print("\nWeather pairs with |rho| > 0.85 (redundant predictors):")

    for a, b, r in sorted(
        high,
        key=lambda t: -abs(t[2])
    ):
        print(f"  {a:22s} {b:22s} {r:+.2f}")

    if not high:
        print("  none")


# ----------------------------------------------------------- 4. time series
def section_timeseries(df):
    print("\n" + "=" * 74)
    print("4. TIME SERIES")
    print("=" * 74)

    fig, axes = plt.subplots(
        len(POLLUTANTS),
        1,
        figsize=(15, 16),
        sharex=True
    )

    for ax, p in zip(axes, POLLUTANTS):

        for st, g in df.groupby("Station"):
            ax.plot(
                g["Date"],
                g[f"{p}_use"],
                lw=.5,
                alpha=.45,
                label=st
            )

        city = df.groupby("Date")[f"{p}_use"].mean()

        ax.plot(
            city.index,
            city.rolling(30, min_periods=10).mean(),
            color="black",
            lw=1.8,
            label="city 30-day mean"
        )

        ax.set_ylabel(
            f"{p}\n{UNITS[p]}"
        )

        ax.grid(alpha=.25)

    axes[0].legend(
        ncol=4,
        fontsize=7
    )

    axes[-1].set_xlabel("Date")

    fig.suptitle(
        "Pollutants by station, usable cells only "
        "(gaps = masked unusable periods)",
        y=.995
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/04_pollutant_timeseries.png",
        dpi=130
    )

    plt.close()

    fig, axes = plt.subplots(
        4,
        1,
        figsize=(15, 11),
        sharex=True
    )

    city = df.groupby("Date")[
        [
            "temp_mean",
            "rain_sum",
            "wind_speed_mean",
            "rh_mean",
        ]
    ].mean()

    for ax, c in zip(
        axes,
        city.columns
    ):

        ax.plot(
            city.index,
            city[c],
            lw=.6,
            color="tab:blue"
        )

        ax.plot(
            city.index,
            city[c].rolling(30, min_periods=10).mean(),
            color="black",
            lw=1.5
        )

        ax.set_ylabel(c)
        ax.grid(alpha=.25)

    fig.suptitle(
        "Weather, city mean",
        y=.995
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/04_weather_timeseries.png",
        dpi=130
    )

    plt.close()

    print(
        "  wrote 04_pollutant_timeseries.png, "
        "04_weather_timeseries.png"
    )

    # The cloud-cover / solar-radiation consistency question raised earlier.
    print(
        "\nCloud cover vs solar radiation by year "
        "(these should move in opposition):"
    )

    yr = df.groupby("year").agg(
        cloud=("cloud_cover_mean", "mean"),
        solar=("solar_rad_sum", "mean")
    )

    yr["rho_within_year"] = df.groupby("year").apply(
        lambda g: sps.spearmanr(
            g["cloud_cover_mean"],
            g["solar_rad_sum"]
        )[0]
    )

    print(yr.round(2).to_string())

    print(
        "  A strong negative within-year rho with a large drift in mean"
    )

    print(
        "  cloud cover but flat mean solar radiation would be internally"
    )

    print(
        "  inconsistent, and would point at the weather series rather than"
    )

    print(
        "  at the atmosphere."
    )


# -------------------------------------------------------------- 5. seasonality
def section_seasonality(df):
    print("\n" + "=" * 74)
    print("5. SEASONAL PATTERNS")
    print("=" * 74)

    seas = df.groupby("season")[
        [f"{p}_use" for p in POLLUTANTS]
    ].mean()

    seas.columns = POLLUTANTS

    order = [
        "Winter",
        "Pre-monsoon",
        "Monsoon",
        "Post-monsoon"
    ]

    seas = seas.reindex(order).round(2)

    print("\nMean concentration by IMD season:")
    print(seas.to_string())

    seas.to_csv(
        f"{OUT_DIR}/05_seasonal_means.csv"
    )

    ratio = (
        seas.max() / seas.min()
    ).round(2)

    print("\nSeasonal amplitude (max season / min season):")
    print(ratio.to_string())

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(16, 8)
    )

    for ax, p in zip(
        axes.ravel(),
        POLLUTANTS
    ):

        data = [
            df.loc[
                df["month"] == m,
                f"{p}_use"
            ].dropna()
            for m in range(1, 13)
        ]

        # FIX:
        # Matplotlib versions that no longer accept `labels=`
        # require `tick_labels=` instead.
        ax.boxplot(
            data,
            tick_labels=[f"{m}" for m in range(1, 13)],
            showfliers=False
        )

        ax.set_title(
            f"{p} ({UNITS[p]})"
        )

        ax.set_xlabel("month")
        ax.grid(alpha=.25)

    fig.suptitle(
        "Monthly distribution, all stations pooled, usable cells only"
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/05_monthly_boxplots.png",
        dpi=130
    )

    plt.close()

    # STL on the city-mean series: separates trend, annual cycle and remainder.
    fig, axes = plt.subplots(
        len(POLLUTANTS),
        1,
        figsize=(14, 16),
        sharex=True
    )

    stl_rows = []

    for ax, p in zip(
        axes,
        POLLUTANTS
    ):

        s = df.groupby("Date")[
            f"{p}_use"
        ].mean()

        s = s.interpolate(
            limit_direction="both"
        )

        res = STL(
            s,
            period=365,
            robust=True
        ).fit()

        ax.plot(
            s.index,
            res.trend,
            color="tab:red",
            lw=1.5,
            label="trend"
        )

        ax.plot(
            s.index,
            res.seasonal + s.mean(),
            color="tab:green",
            lw=.7,
            alpha=.7,
            label="seasonal"
        )

        ax.set_ylabel(p)
        ax.grid(alpha=.25)

        var_s = np.var(res.seasonal)
        var_r = np.var(res.resid)

        stl_rows.append({
            "pollutant": p,
            "seasonal_strength": round(
                var_s / (var_s + var_r),
                3
            ),
            "trend_change": round(
                res.trend.iloc[-1] - res.trend.iloc[0],
                2
            ),
        })

    axes[0].legend(fontsize=7)

    fig.suptitle(
        "STL decomposition, city-mean series",
        y=.995
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/05_stl_decomposition.png",
        dpi=130
    )

    plt.close()

    stl = pd.DataFrame(stl_rows)

    print("\nSTL on city-mean series:")
    print(stl.to_string(index=False))

    stl.to_csv(
        f"{OUT_DIR}/05_stl_summary.csv",
        index=False
    )

    print(
        "  seasonal_strength near 1 means the annual cycle dominates the"
    )

    print(
        "  remainder; a forecasting model must capture it explicitly."
    )

    print(
        "  NOTE: STL needs a gapless index, so masked days were interpolated"
    )

    print(
        "  FOR THIS PLOT ONLY. Nothing else in this file uses interpolation."
    )


# ------------------------------------------------- 6. pollutant-weather shape
def section_relationships(df):
    print("\n" + "=" * 74)
    print("6. POLLUTANT-WEATHER RELATIONSHIPS (binned, not fitted)")
    print("=" * 74)

    pairs = [
        ("PM2.5", "wind_speed_mean"),
        ("PM2.5", "rain_sum"),
        ("PM2.5", "rh_mean"),
        ("PM10", "wind_speed_mean"),
        ("Ozone", "solar_rad_sum"),
        ("Ozone", "temp_max"),
        ("NO2", "wind_speed_mean"),
        ("CO", "wind_speed_mean"),
        ("SO2", "wind_speed_mean"),
    ]

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(15, 12)
    )

    for ax, (p, w) in zip(
        axes.ravel(),
        pairs
    ):

        d = df[
            [f"{p}_use", w]
        ].dropna()

        ax.scatter(
            d[w],
            d[f"{p}_use"],
            s=3,
            alpha=.12,
            color="tab:blue"
        )

        try:
            b = pd.qcut(
                d[w],
                10,
                duplicates="drop"
            )

            med = d.groupby(
                b,
                observed=True
            ).median()

            ax.plot(
                med[w],
                med[f"{p}_use"],
                color="crimson",
                lw=2,
                marker="o",
                ms=4
            )

        except ValueError:
            pass

        ax.set_xlabel(w)
        ax.set_ylabel(
            f"{p} ({UNITS[p]})"
        )

        ax.grid(alpha=.25)

    fig.suptitle(
        "Red line = median within decile of the weather variable. "
        "Association only; no causal claim.",
        y=.995
    )

    plt.tight_layout()

    plt.savefig(
        f"{OUT_DIR}/06_relationships.png",
        dpi=130
    )

    plt.close()

    print(
        "  wrote 06_relationships.png"
    )

    print(
        "\nMedian PM2.5 by rainfall band (µg/m³):"
    )

    bands = pd.cut(
        df["rain_sum"],
        [-.01, .1, 1, 5, 20, 1000],
        labels=[
            "dry",
            "<1mm",
            "1-5mm",
            "5-20mm",
            ">20mm"
        ]
    )

    print(
        df.groupby(
            bands,
            observed=True
        )["PM2.5_use"]
        .agg(["count", "median"])
        .round(2)
        .to_string()
    )


# ------------------------------------------------------------------ 7. outliers
def section_outliers(df):
    print("\n" + "=" * 74)
    print("7. OUTLIER ANALYSIS (post-repair, usable cells only)")
    print("=" * 74)

    print(
        "Tukey fences on the log scale, since the pollutants are strongly"
    )

    print(
        "right-skewed and fences on the raw scale would flag the entire"
    )

    print(
        "upper tail of a legitimately skewed distribution.\n"
    )

    rows = []

    for p in POLLUTANTS:

        s = df[
            f"{p}_use"
        ].dropna()

        ls = np.log1p(s)

        q1 = ls.quantile(.25)
        q3 = ls.quantile(.75)

        iqr = q3 - q1

        hi = ls > q3 + 3 * iqr
        lo = ls < q1 - 3 * iqr

        rows.append({
            "pollutant": p,
            "n": len(s),
            "extreme_high": int(hi.sum()),
            "extreme_low": int(lo.sum()),
            "pct": round(
                (hi.sum() + lo.sum()) / len(s) * 100,
                2
            ),
            "max": round(
                s.max(),
                2
            ),
            "p99": round(
                s.quantile(.99),
                2
            ),
        })

    tab = pd.DataFrame(rows)

    print(
        tab.to_string(index=False)
    )

    tab.to_csv(
        f"{OUT_DIR}/07_outliers.csv",
        index=False
    )

    print(
        "\n  These are candidates for genuine pollution episodes, not errors:"
    )

    print(
        "  instrument faults were already removed by the repair pass. Check"
    )

    print(
        "  whether they cluster on the same dates across stations, which"
    )

    print(
        "  would indicate a real city-wide event."
    )

    dates = {}

    for p in POLLUTANTS:

        s = df[
            ["Date", "Station", f"{p}_use"]
        ].dropna()

        ls = np.log1p(
            s[f"{p}_use"]
        )

        q1 = ls.quantile(.25)
        q3 = ls.quantile(.75)

        top = s[
            ls > q3 + 3 * (q3 - q1)
        ]

        for d in top["Date"].unique():

            dates.setdefault(
                pd.Timestamp(d),
                set()
            ).add(p)

    multi = {
        d: v
        for d, v in dates.items()
        if len(v) >= 2
    }

    print(
        f"\n  Dates with extremes in 2+ pollutants: {len(multi)}"
    )

    for d in sorted(multi)[:12]:
        print(
            f"    {d.date()}  {sorted(multi[d])}"
        )


# --------------------------------------------------------------- 8. stationarity
def longest_usable_block(s):
    ok = s.notna().values

    best = (0, 0, 0)

    i = 0

    while i < len(ok):

        if ok[i]:

            j = i

            while (
                j + 1 < len(ok)
                and ok[j + 1]
            ):
                j += 1

            if j - i + 1 > best[2]:
                best = (
                    i,
                    j,
                    j - i + 1
                )

            i = j + 1

        else:
            i += 1

    return best


def section_stationarity(df):
    print("\n" + "=" * 74)
    print("8. STATIONARITY")
    print("=" * 74)

    print(
        "ADF (H0: unit root) and KPSS (H0: stationary) on the longest"
    )

    print(
        "CONTIGUOUS usable run per station-pollutant. Contiguity matters:"
    )

    print(
        "dropping masked days and closing the gap would splice together"
    )

    print(
        "dates that are not adjacent and corrupt the lag structure both"
    )

    print(
        "tests depend on.\n"
    )

    rows = []

    for p in POLLUTANTS:

        for st, g in df.groupby("Station"):

            s = (
                g.sort_values("Date")[f"{p}_use"]
                .reset_index(drop=True)
            )

            i, j, n = longest_usable_block(s)

            if n < MIN_BLOCK:

                rows.append({
                    "pollutant": p,
                    "station": st,
                    "n": n,
                    "adf_p": np.nan,
                    "kpss_p": np.nan,
                    "verdict": f"block too short ({n}d)",
                })

                continue

            block = s.iloc[
                i:j + 1
            ]

            try:
                adf_p = adfuller(
                    block,
                    autolag="AIC"
                )[1]

            except Exception:
                adf_p = np.nan

            try:
                kpss_p = kpss(
                    block,
                    regression="c",
                    nlags="auto"
                )[1]

            except Exception:
                kpss_p = np.nan

            adf_stat = adf_p < .05
            kpss_stat = kpss_p > .05

            if adf_stat and kpss_stat:
                v = "stationary"

            elif not adf_stat and not kpss_stat:
                v = "non-stationary"

            elif adf_stat and not kpss_stat:
                v = "conflict: trend-stationary?"

            else:
                v = "conflict: inconclusive"

            rows.append({
                "pollutant": p,
                "station": st,
                "n": n,
                "adf_p": round(adf_p, 4),
                "kpss_p": round(kpss_p, 4),
                "verdict": v,
            })

    tab = pd.DataFrame(rows)

    print(
        tab.to_string(index=False)
    )

    tab.to_csv(
        f"{OUT_DIR}/08_stationarity.csv",
        index=False
    )

    print("\nVerdict counts:")

    print(
        tab["verdict"].value_counts().to_string()
    )

    print(
        "\n  KPSS p-values are reported by statsmodels within a lookup table"
    )

    print(
        "  and clip at 0.01 and 0.10, so a p of exactly 0.01 means '<=0.01'."
    )

    print(
        "  A conflict is not a failure: with a strong annual cycle both"
    )

    print(
        "  tests are being asked the wrong question, since neither treats"
    )

    print(
        "  seasonality as anything other than non-stationarity. Read this"
    )

    print(
        "  table alongside the STL seasonal strengths in section 5."
    )


# ----------------------------------------------------------------------------
def main():

    os.makedirs(
        OUT_DIR,
        exist_ok=True
    )

    df = load()

    print(
        f"Loaded {IN_DATA}  {df.shape}"
    )

    print(
        f"{df['Date'].min().date()} to "
        f"{df['Date'].max().date()}, "
        f"{df['Station'].nunique()} stations"
    )

    section_missingness(df)
    section_descriptives(df)
    section_correlation(df)
    section_timeseries(df)
    section_seasonality(df)
    section_relationships(df)
    section_outliers(df)
    section_stationarity(df)

    print("\n" + "=" * 74)
    print(
        f"Done. Figures and tables in {OUT_DIR}/"
    )
    print("=" * 74)

    for f in sorted(
        os.listdir(OUT_DIR)
    ):
        print("  ", f)

    print(
        "\nNo file outside eda_outputs/ was created or modified."
    )


if __name__ == "__main__":
    main()



Loaded Bengaluru_AQ_Weather_daily_v3_noPBL.csv  (6576, 73)
2023-07-01 to 2026-06-30, 6 stations

1. DATA AVAILABILITY AFTER MASKING
pollutant  measured_%  estimated_%  unusable_%  n_usable
    PM2.5        89.8          6.4         3.9      6322
     PM10        90.9          6.1         2.9      6384
      NO2        74.9          7.1        18.0      5395
      SO2        91.5          6.8         1.7      6467
       CO        69.7          8.1        22.2      5118
    Ozone        85.7          8.0         6.3      6161

Usable station-days by station x pollutant:
                     PM2.5  PM10   NO2   SO2   CO  Ozone
Station                                                 
BTM Layout            1096  1096   994  1071  915   1096
Hebbal                 975  1034   479  1040  915   1008
Jayanagar 5th Block   1096  1096   863  1088  849   1002
Kasturi Nagar         1084   966  1004  1096  915    914
Peenya                 975  1096  1091  1083  762   1096
Silk Board            109

In [2]:
import os

project = "Bengaluru-AQI-Forecasting"
os.makedirs(project, exist_ok=True)

print(os.path.abspath(project))


C:\Users\Akansha\Bengaluru-AQI-Forecasting


In [3]:
import os

for item in os.listdir("."):
    if os.path.isdir(item):
        print("[FOLDER]", item)

[FOLDER] .cache
[FOLDER] .conda
[FOLDER] .continuum
[FOLDER] .copilot
[FOLDER] .idlerc
[FOLDER] .ipynb_checkpoints
[FOLDER] .ipython
[FOLDER] .jupyter
[FOLDER] .kaggle
[FOLDER] .keras
[FOLDER] .matplotlib
[FOLDER] .ms-ad
[FOLDER] .thumbnails
[FOLDER] .vscode
[FOLDER] .vscode-shared
[FOLDER] .wdm
[FOLDER] 2548004
[FOLDER] ado
[FOLDER] akan04
[FOLDER] akzzgotrizz_proj
[FOLDER] anaconda3
[FOLDER] AppData
[FOLDER] Application Data
[FOLDER] Bengaluru-AQI-Forecasting
[FOLDER] Contacts
[FOLDER] Cookies
[FOLDER] data
[FOLDER] data_analysis_project
[FOLDER] Desktop
[FOLDER] Downloads
[FOLDER] eda_outputs
[FOLDER] Favorites
[FOLDER] FinalExam
[FOLDER] IdeaProjects
[FOLDER] jennie
[FOLDER] KeyFrames
[FOLDER] lab4_proj
[FOLDER] lab5_proj
[FOLDER] lab7_proj
[FOLDER] lab8_proj
[FOLDER] lalaland_proj
[FOLDER] Links
[FOLDER] Local Settings
[FOLDER] Machine Learning
[FOLDER] Microsoft
[FOLDER] Music
[FOLDER] myenv
[FOLDER] myproject
[FOLDER] NetHood
[FOLDER] node_modules
[FOLDER] nptel
[FOLDER] OneDriv